In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
from IPython.display import display

# =========================================================
# 0. ROOTS
# =========================================================
PROJECT_ROOT = Path("/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability")
RETINAL_RECORD_ROOT = PROJECT_ROOT / "06_Data_Records" / "Retinal_DR"

BASELINE_ROOT = RETINAL_RECORD_ROOT / "Frozen_Representation_Baseline_v0.1"
KILL_TEST_ROOT = RETINAL_RECORD_ROOT / "Locality_Kill_Test_v0.1"
FINE_STRUCTURE_ROOT = RETINAL_RECORD_ROOT / "Fine_Structure_Dissection_v0.1"

EXTERNAL_ROOT = RETINAL_RECORD_ROOT / "External_Replication_Protocol_v0.1"
EXTERNAL_ROOT.mkdir(parents=True, exist_ok=True)

# =========================================================
# 1. SEALED INPUT PATHS
# =========================================================
BASELINE_DECISION_PATH = BASELINE_ROOT / "Frozen_Representation_Baseline_v0.1_Decision.json"
KILL_TEST_DECISION_PATH = KILL_TEST_ROOT / "Locality_Kill_Test_v0.1_Decision.json"
FINE_STRUCTURE_DECISION_PATH = FINE_STRUCTURE_ROOT / "Fine_Structure_Dissection_v0.1_Final_Decision.json"

REQUIRED_INPUTS = [
    BASELINE_DECISION_PATH,
    KILL_TEST_DECISION_PATH,
    FINE_STRUCTURE_DECISION_PATH,
]

for p in REQUIRED_INPUTS:
    assert p.exists(), f"Missing required sealed input: {p}"

# =========================================================
# 2. HELPERS
# =========================================================
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def extract_decision_string(obj):
    if isinstance(obj, str):
        return obj
    if isinstance(obj, dict):
        for key in ["decision", "Decision", "final_decision", "FinalDecision"]:
            if key in obj:
                return str(obj[key])
    raise KeyError(f"Cannot find decision string in object: {obj}")

baseline_decision_obj = load_json(BASELINE_DECISION_PATH)
kill_test_decision_obj = load_json(KILL_TEST_DECISION_PATH)
fine_structure_decision_obj = load_json(FINE_STRUCTURE_DECISION_PATH)

baseline_decision = extract_decision_string(baseline_decision_obj)
kill_test_decision = extract_decision_string(kill_test_decision_obj)
fine_structure_decision = extract_decision_string(fine_structure_decision_obj)

# =========================================================
# 3. GATING ASSERTIONS
# =========================================================
assert baseline_decision == "PASS_BASELINE_ADVANCE_TO_LOCALITY_KILL_TEST", (
    f"Unexpected baseline decision: {baseline_decision}"
)

assert kill_test_decision == "PASS_FINE_SCALE_ACHROMATIC_STRUCTURE_DEPENDENCY", (
    f"Unexpected kill-test decision: {kill_test_decision}"
)

assert fine_structure_decision == "PASS_DEEPDRID_FINE_STRUCTURE_DISSECTION_ADVANCE_TO_EXTERNAL_REPLICATION", (
    f"Unexpected fine-structure decision: {fine_structure_decision}"
)

# =========================================================
# 4. OUTPUT PATHS
# =========================================================
PROTOCOL_PATH = EXTERNAL_ROOT / "External_Replication_Protocol_v0.1.csv"
REGISTRY_TEMPLATE_PATH = EXTERNAL_ROOT / "External_Candidate_Registry_Template_v0.1.csv"
BOOTSTRAP_PATH = EXTERNAL_ROOT / "External_Replication_Bootstrap.json"

# =========================================================
# 5. PRE-REGISTERED PROTOCOL TABLE
# =========================================================
protocol_df = pd.DataFrame([
    {
        "stage": 1,
        "analysis": "Sealed-input import",
        "status": "pre-registered",
        "primary_question": "Do prior sealed decisions authorize external replication?",
        "planned_outputs": "Bootstrap json; protocol csv",
        "decision_rule": "Must pass all gating assertions from baseline, locality kill test, and fine-structure dissection."
    },
    {
        "stage": 2,
        "analysis": "External candidate registry",
        "status": "pre-registered",
        "primary_question": "Which external datasets are eligible for patient-within-patient eye-local replication?",
        "planned_outputs": "Candidate registry csv",
        "decision_rule": "Dataset must support patient ID, left/right eye identity, and eye-level DR grade."
    },
    {
        "stage": 3,
        "analysis": "Bilateral availability audit",
        "status": "pre-registered",
        "primary_question": "How many patients have both eyes available?",
        "planned_outputs": "Dataset audit tables",
        "decision_rule": "Proceed only if bilateral patients can be unambiguously identified."
    },
    {
        "stage": 4,
        "analysis": "Unequal-grade pair audit",
        "status": "pre-registered",
        "primary_question": "How many bilateral patients have unequal DR grades between eyes?",
        "planned_outputs": "Unequal-grade audit tables",
        "decision_rule": "Primary replication cohort requires bilateral unequal-grade patients."
    },
    {
        "stage": 5,
        "analysis": "Frozen encoder extraction",
        "status": "pre-registered",
        "primary_question": "Can the same frozen representation be extracted on the external dataset?",
        "planned_outputs": "Embedding arrays and index tables",
        "decision_rule": "Use the same sealed encoder family and preprocessing as prior notebook."
    },
    {
        "stage": 6,
        "analysis": "Transferred sealed condition tests",
        "status": "pre-registered",
        "primary_question": "Do sealed conditions replicate externally?",
        "planned_outputs": "Condition summary tables",
        "decision_rule": "Transfer only sealed conditions: clean, grayscale, strong blur, central detail preserved, peripheral detail preserved, global mix 50."
    },
    {
        "stage": 7,
        "analysis": "External evidence table and final decision",
        "status": "pre-registered",
        "primary_question": "Does the external dataset replicate the eye-local severity direction signal?",
        "planned_outputs": "Evidence table; decision json; report md",
        "decision_rule": "Decision based on the pre-registered primary endpoint: within-patient eye-severity direction prediction."
    },
])

protocol_df.to_csv(PROTOCOL_PATH, index=False)

bootstrap_obj = {
    "baseline_decision": baseline_decision,
    "kill_test_decision": kill_test_decision,
    "fine_structure_decision": fine_structure_decision,
    "external_output_root": str(EXTERNAL_ROOT),
    "authorized_primary_endpoint": (
        "Within the same patient, predict which eye has the higher DR severity grade "
        "using frozen image representations."
    ),
    "sealed_transfer_conditions": [
        "clean",
        "grayscale",
        "strong_blur",
        "central_detail_preserved",
        "peripheral_detail_preserved",
        "global_mix_50"
    ]
}

with open(BOOTSTRAP_PATH, "w", encoding="utf-8") as f:
    json.dump(bootstrap_obj, f, indent=2, ensure_ascii=False)

print("============== EXTERNAL REPLICATION BOOTSTRAP ==============")
print("Baseline decision:", baseline_decision)
print("Kill-test decision:", kill_test_decision)
print("Fine-structure decision:", fine_structure_decision)
print("\nExternal output root:")
print(EXTERNAL_ROOT)

print("\nSaved:")
print(PROTOCOL_PATH)
print(BOOTSTRAP_PATH)

print("\nPre-registered protocol:")
display(protocol_df)

Mounted at /content/drive
============== EXTERNAL REPLICATION BOOTSTRAP ==============
Baseline decision: PASS_BASELINE_ADVANCE_TO_LOCALITY_KILL_TEST
Kill-test decision: PASS_FINE_SCALE_ACHROMATIC_STRUCTURE_DEPENDENCY
Fine-structure decision: PASS_DEEPDRID_FINE_STRUCTURE_DISSECTION_ADVANCE_TO_EXTERNAL_REPLICATION

External output root:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/External_Replication_Protocol_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/External_Replication_Bootstrap.json

Pre-registered protocol:


,stage,analysis,status,primary_question,planned_outputs,decision_rule
0,1,Sealed-input import,pre-registered,Do prior sealed decisions authorize external r...,Bootstrap json; protocol csv,"Must pass all gating assertions from baseline,..."
1,2,External candidate registry,pre-registered,Which external datasets are eligible for patie...,Candidate registry csv,"Dataset must support patient ID, left/right ey..."
2,3,Bilateral availability audit,pre-registered,How many patients have both eyes available?,Dataset audit tables,Proceed only if bilateral patients can be unam...
3,4,Unequal-grade pair audit,pre-registered,How many bilateral patients have unequal DR gr...,Unequal-grade audit tables,Primary replication cohort requires bilateral ...
4,5,Frozen encoder extraction,pre-registered,Can the same frozen representation be extracte...,Embedding arrays and index tables,Use the same sealed encoder family and preproc...
5,6,Transferred sealed condition tests,pre-registered,Do sealed conditions replicate externally?,Condition summary tables,"Transfer only sealed conditions: clean, graysc..."
6,7,External evidence table and final decision,pre-registered,Does the external dataset replicate the eye-lo...,Evidence table; decision json; report md,Decision based on the pre-registered primary e...


In [2]:
# =========================================================
# External candidate registry template
# =========================================================
candidate_registry_columns = [
    "dataset_name",
    "dataset_root",
    "metadata_path",
    "image_root",
    "patient_id_col",
    "eye_side_col",
    "dr_grade_col",
    "image_path_col",
    "can_identify_patient",
    "can_identify_eye",
    "has_eye_level_grade",
    "has_bilateral_patients",
    "supports_unequal_grade_pairs",
    "notes"
]

candidate_registry_df = pd.DataFrame(columns=candidate_registry_columns)

candidate_registry_df.to_csv(REGISTRY_TEMPLATE_PATH, index=False)

print("============== EXTERNAL CANDIDATE REGISTRY TEMPLATE ==============")
print("Saved:")
print(REGISTRY_TEMPLATE_PATH)

display(candidate_registry_df)

============== EXTERNAL CANDIDATE REGISTRY TEMPLATE ==============
Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/External_Candidate_Registry_Template_v0.1.csv


,dataset_name,dataset_root,metadata_path,image_root,patient_id_col,eye_side_col,dr_grade_col,image_path_col,can_identify_patient,can_identify_eye,has_eye_level_grade,has_bilateral_patients,supports_unequal_grade_pairs,notes


In [4]:
# =========================================================
# Helpers for candidate dataset audit
# =========================================================

LEFT_TOKENS = {"l", "left", "os", "le", "left_eye"}
RIGHT_TOKENS = {"r", "right", "od", "re", "right_eye"}

def canonical_eye_label(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in LEFT_TOKENS:
        return "left"
    if s in RIGHT_TOKENS:
        return "right"
    if s.startswith("l"):
        return "left"
    if s.startswith("r"):
        return "right"
    return np.nan

def load_external_metadata_csv(
    metadata_path,
    dataset_name,
    patient_id_col,
    eye_side_col,
    dr_grade_col,
    image_path_col
):
    df = pd.read_csv(metadata_path)

    required_source_cols = [
        patient_id_col,
        eye_side_col,
        dr_grade_col,
        image_path_col,
    ]
    missing = [c for c in required_source_cols if c not in df.columns]
    assert not missing, f"Missing columns in source metadata: {missing}"

    out = pd.DataFrame({
        "dataset_name": dataset_name,
        "patient_id": df[patient_id_col].astype(str),
        "eye_side_raw": df[eye_side_col],
        "eye_side": df[eye_side_col].map(canonical_eye_label),
        "dr_grade": pd.to_numeric(df[dr_grade_col], errors="coerce"),
        "image_path": df[image_path_col].astype(str),
    })

    out = out.dropna(subset=["patient_id", "eye_side", "dr_grade", "image_path"]).copy()
    out["dr_grade"] = out["dr_grade"].astype(int)

    return out

def audit_external_dataset_structure(standardized_df):
    # 先检查同一 patient-eye 下是否出现多个 grade
    eye_grade_nunique = (
        standardized_df
        .groupby(["patient_id", "eye_side"])["dr_grade"]
        .nunique()
        .rename("n_unique_grades")
        .reset_index()
    )

    inconsistent_eye_units = eye_grade_nunique[eye_grade_nunique["n_unique_grades"] > 1].copy()

    # 每个 patient-eye 暂时取第一条记录，后续真的要跑模型时再决定具体选图策略
    eye_level_df = (
        standardized_df
        .sort_values(["patient_id", "eye_side", "image_path"])
        .groupby(["patient_id", "eye_side"], as_index=False)
        .first()
    )

    eye_count_per_patient = eye_level_df.groupby("patient_id")["eye_side"].nunique()
    bilateral_patient_ids = eye_count_per_patient[eye_count_per_patient == 2].index.tolist()

    bilateral_eye_df = eye_level_df[eye_level_df["patient_id"].isin(bilateral_patient_ids)].copy()

    bilateral_wide = (
        bilateral_eye_df
        .pivot(index="patient_id", columns="eye_side", values="dr_grade")
        .reset_index()
    )

    if "left" not in bilateral_wide.columns:
        bilateral_wide["left"] = np.nan
    if "right" not in bilateral_wide.columns:
        bilateral_wide["right"] = np.nan

    bilateral_wide["absolute_grade_gap"] = (bilateral_wide["left"] - bilateral_wide["right"]).abs()
    unequal_grade_df = bilateral_wide[bilateral_wide["absolute_grade_gap"] > 0].copy()

    summary = pd.DataFrame([{
        "image_rows_after_cleaning": int(len(standardized_df)),
        "eye_rows_after_dedup": int(len(eye_level_df)),
        "unique_patients": int(eye_level_df["patient_id"].nunique()),
        "bilateral_patients": int(len(bilateral_patient_ids)),
        "unequal_grade_patients": int(len(unequal_grade_df)),
        "inconsistent_patient_eye_units": int(len(inconsistent_eye_units)),
    }])

    grade_gap_distribution = (
        unequal_grade_df["absolute_grade_gap"]
        .value_counts()
        .sort_index()
        .rename_axis("absolute_grade_gap")
        .reset_index(name="count")
    )

    return {
        "summary": summary,
        "eye_level_df": eye_level_df,
        "bilateral_wide": bilateral_wide,
        "unequal_grade_df": unequal_grade_df,
        "grade_gap_distribution": grade_gap_distribution,
        "inconsistent_eye_units": inconsistent_eye_units,
    }

print("External audit helpers loaded.")

External audit helpers loaded.


In [5]:
#@title 04. Freeze EyePACS 2015 as the primary external candidate

from pathlib import Path
import json
import pandas as pd


# ============================================================
# 1. Candidate identity and planned local structure
# ============================================================

EYEPACS_DATASET_HANDLE = (
    "tanlikesmath/diabetic-retinopathy-resized"
)

EYEPACS_DATASET_NAME = (
    "EyePACS_2015_Resized_Cropped"
)

EYEPACS_DATASET_ROOT = (
    PROJECT_ROOT
    / "02_Dataset_Map"
    / "EyePACS_2015_External_Replication"
)

EYEPACS_METADATA_ROOT = (
    EYEPACS_DATASET_ROOT
    / "metadata"
)

EYEPACS_IMAGE_ROOT = (
    EYEPACS_DATASET_ROOT
    / "resized_train_cropped"
)

EYEPACS_METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EYEPACS_IMAGE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


PLANNED_METADATA_PATH = (
    EYEPACS_METADATA_ROOT
    / "trainLabels.csv"
)

CANDIDATE_REGISTRY_PATH = (
    EXTERNAL_ROOT
    / "External_Candidate_Registry_v0.1.csv"
)

CANDIDATE_DECISION_PATH = (
    EXTERNAL_ROOT
    / "External_Primary_Candidate_Decision_v0.1.json"
)


# ============================================================
# 2. Pre-register minimum feasibility requirements
#
# These rules are frozen before reading the EyePACS labels.
# ============================================================

MINIMUM_UNEQUAL_GRADE_PATIENTS = 180
MINIMUM_HELD_OUT_PATIENTS = 45

candidate_decision = {
    "primary_candidate": EYEPACS_DATASET_NAME,
    "kaggle_dataset_handle": EYEPACS_DATASET_HANDLE,
    "dataset_subset": (
        "EyePACS 2015 labelled training images only"
    ),
    "planned_image_variant": (
        "resized_train_cropped"
    ),
    "primary_endpoint": (
        "Within the same patient, predict which eye "
        "has the higher DR grade."
    ),
    "minimum_unequal_grade_patients": (
        MINIMUM_UNEQUAL_GRADE_PATIENTS
    ),
    "minimum_held_out_patients": (
        MINIMUM_HELD_OUT_PATIENTS
    ),
    "proceed_rule": (
        "Proceed only if at least 180 bilateral "
        "unequal-grade patients are available, allowing "
        "a held-out cohort of at least 45 patients, and "
        "both left-higher and right-higher directions "
        "are represented."
    ),
    "excluded_subset": (
        "APTOS 2019 because subject identity and "
        "left/right eye identity are unavailable."
    ),
    "provenance_boundary": (
        "This is a public resized/cropped derivative of "
        "the EyePACS 2015 Kaggle data, not the untouched "
        "original-resolution archive."
    ),
}


with open(
    CANDIDATE_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        candidate_decision,
        file,
        indent=2,
    )


# ============================================================
# 3. Candidate registry row
# ============================================================

candidate_registry_df = pd.DataFrame(
    [
        {
            "dataset_name": (
                EYEPACS_DATASET_NAME
            ),
            "dataset_root": str(
                EYEPACS_DATASET_ROOT
            ),
            "metadata_path": str(
                PLANNED_METADATA_PATH
            ),
            "image_root": str(
                EYEPACS_IMAGE_ROOT
            ),
            "patient_id_col": (
                "derived from image filename"
            ),
            "eye_side_col": (
                "derived from image filename"
            ),
            "dr_grade_col": "level",
            "image_path_col": "image",
            "can_identify_patient": True,
            "can_identify_eye": True,
            "has_eye_level_grade": True,
            "has_bilateral_patients": (
                "pending metadata audit"
            ),
            "supports_unequal_grade_pairs": (
                "pending metadata audit"
            ),
            "notes": (
                "Use EyePACS 2015 labelled training "
                "subset only. Filename pattern is "
                "patientID_left/right. Exclude APTOS "
                "2019. Metadata audit precedes image "
                "download."
            ),
        }
    ]
)


candidate_registry_df.to_csv(
    CANDIDATE_REGISTRY_PATH,
    index=False,
)


print(
    "============== PRIMARY EXTERNAL "
    "CANDIDATE FROZEN =============="
)

print("Dataset:", EYEPACS_DATASET_NAME)
print("Kaggle handle:", EYEPACS_DATASET_HANDLE)

print("\nPre-registered feasibility rule:")
print(candidate_decision["proceed_rule"])

print("\nPlanned metadata path:")
print(PLANNED_METADATA_PATH)

print("\nPlanned image root:")
print(EYEPACS_IMAGE_ROOT)

print("\nSaved:")
print(CANDIDATE_REGISTRY_PATH)
print(CANDIDATE_DECISION_PATH)

display(candidate_registry_df)

============== PRIMARY EXTERNAL CANDIDATE FROZEN ==============
Dataset: EyePACS_2015_Resized_Cropped
Kaggle handle: tanlikesmath/diabetic-retinopathy-resized

Pre-registered feasibility rule:
Proceed only if at least 180 bilateral unequal-grade patients are available, allowing a held-out cohort of at least 45 patients, and both left-higher and right-higher directions are represented.

Planned metadata path:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/EyePACS_2015_External_Replication/metadata/trainLabels.csv

Planned image root:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/EyePACS_2015_External_Replication/resized_train_cropped

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/External_Candidate_Registry_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/External_Prim

,dataset_name,dataset_root,metadata_path,image_root,patient_id_col,eye_side_col,dr_grade_col,image_path_col,can_identify_patient,can_identify_eye,has_eye_level_grade,has_bilateral_patients,supports_unequal_grade_pairs,notes
0,EyePACS_2015_Resized_Cropped,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,derived from image filename,derived from image filename,level,image,True,True,True,pending metadata audit,pending metadata audit,Use EyePACS 2015 labelled training subset only...


In [8]:
#@title 05. Anonymous EyePACS metadata download and feasibility audit

from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd


# ============================================================
# 1. Resolve Google Drive and paths
# ============================================================

if Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

elif Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")

else:
    raise FileNotFoundError(
        "Google Drive is not mounted. "
        "Please rerun the first notebook cell."
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_RECORD_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

EXTERNAL_ROOT = (
    RETINAL_RECORD_ROOT
    / "External_Replication_Protocol_v0.1"
)

EXTERNAL_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EYEPACS_DATASET_HANDLE = (
    "tanlikesmath/diabetic-retinopathy-resized"
)

EYEPACS_DATASET_NAME = (
    "EyePACS_2015_Resized_Cropped"
)

EYEPACS_DATASET_ROOT = (
    PROJECT_ROOT
    / "02_Dataset_Map"
    / "EyePACS_2015_External_Replication"
)

EYEPACS_METADATA_ROOT = (
    EYEPACS_DATASET_ROOT
    / "metadata"
)

EYEPACS_IMAGE_ROOT = (
    EYEPACS_DATASET_ROOT
    / "resized_train_cropped"
)

EYEPACS_METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EYEPACS_IMAGE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


PLANNED_METADATA_PATH = (
    EYEPACS_METADATA_ROOT
    / "trainLabels.csv"
)

CANDIDATE_REGISTRY_PATH = (
    EXTERNAL_ROOT
    / "External_Candidate_Registry_v0.1.csv"
)

MINIMUM_UNEQUAL_GRADE_PATIENTS = 180
MINIMUM_HELD_OUT_PATIENTS = 45


print(
    "================ EYEPACS ANONYMOUS "
    "METADATA SETUP ================"
)

print("Drive root:")
print(DRIVE_ROOT)

print("\nPermanent metadata path:")
print(PLANNED_METADATA_PATH)


# ============================================================
# 2. Anonymous KaggleHub single-file download
# ============================================================

if PLANNED_METADATA_PATH.is_file():

    print(
        "\nExisting trainLabels.csv detected."
    )

    print(
        "Skipping download:"
    )

    print(PLANNED_METADATA_PATH)

else:

    print(
        "\nInstalling KaggleHub..."
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "kagglehub",
        ],
        check=True,
    )

    import kagglehub

    print(
        "\nAttempting anonymous single-file download..."
    )

    try:
        downloaded_result = (
            kagglehub.dataset_download(
                EYEPACS_DATASET_HANDLE,
                path="trainLabels.csv",
                output_dir=str(
                    EYEPACS_METADATA_ROOT
                ),
                force_download=True,
            )
        )

    except Exception as error:
        raise RuntimeError(
            "Anonymous Kaggle download was rejected.\n"
            "This means Kaggle is requiring account "
            "authentication or consent for this resource.\n"
            "No kaggle.json was created or exposed."
        ) from error


    print(
        "\nKaggleHub returned:"
    )

    print(downloaded_result)


    # Search only inside our permanent Drive metadata folder.
    metadata_candidates = list(
        EYEPACS_METADATA_ROOT.rglob(
            "trainLabels.csv"
        )
    )

    if not metadata_candidates:
        returned_path = Path(
            str(downloaded_result)
        )

        if (
            returned_path.is_file()
            and returned_path.name
            == "trainLabels.csv"
        ):
            metadata_candidates = [
                returned_path
            ]

        elif returned_path.is_dir():
            metadata_candidates = list(
                returned_path.rglob(
                    "trainLabels.csv"
                )
            )


    if not metadata_candidates:
        raise FileNotFoundError(
            "KaggleHub completed, but "
            "trainLabels.csv could not be located."
        )


    source_metadata_path = (
        metadata_candidates[0]
    )


    if (
        source_metadata_path.resolve()
        != PLANNED_METADATA_PATH.resolve()
    ):
        shutil.copy2(
            source_metadata_path,
            PLANNED_METADATA_PATH,
        )


    assert PLANNED_METADATA_PATH.is_file()

    print(
        "\nMetadata permanently saved to Drive:"
    )

    print(PLANNED_METADATA_PATH)


# ============================================================
# 3. Load and validate raw metadata
# ============================================================

raw_labels = pd.read_csv(
    PLANNED_METADATA_PATH
)


print(
    "\n================ RAW METADATA "
    "AUDIT ================"
)

print(
    "Raw metadata shape:",
    raw_labels.shape,
)

print(
    "Raw columns:",
    raw_labels.columns.tolist(),
)


required_columns = [
    "image",
    "level",
]

missing_columns = [
    column
    for column in required_columns
    if column not in raw_labels.columns
]

if missing_columns:
    raise ValueError(
        "Missing required metadata columns: "
        f"{missing_columns}"
    )


# ============================================================
# 4. Parse patient and eye from filename
# ============================================================

raw_labels["image_stem"] = (
    raw_labels["image"]
    .astype(str)
    .map(
        lambda value: Path(value).stem
    )
)


parsed_filename = (
    raw_labels["image_stem"]
    .str.extract(
        r"^(?P<patient_id>.+)_"
        r"(?P<eye_side>left|right)$"
    )
)


unparsed_mask = (
    parsed_filename[
        [
            "patient_id",
            "eye_side",
        ]
    ]
    .isna()
    .any(axis=1)
)


print(
    "\nUnparsed filename rows:",
    int(unparsed_mask.sum()),
)


if unparsed_mask.any():

    display(
        raw_labels.loc[
            unparsed_mask,
            ["image"],
        ].head(20)
    )

    raise ValueError(
        "Some filenames do not follow "
        "patientID_left/right."
    )


standardized_eye_pacs = pd.DataFrame(
    {
        "dataset_name": (
            EYEPACS_DATASET_NAME
        ),
        "patient_id": (
            parsed_filename[
                "patient_id"
            ].astype(str)
        ),
        "eye_side": (
            parsed_filename[
                "eye_side"
            ].astype(str)
        ),
        "dr_grade": pd.to_numeric(
            raw_labels["level"],
            errors="raise",
        ).astype(int),
        "image_id": (
            raw_labels[
                "image_stem"
            ].astype(str)
        ),
    }
)


standardized_eye_pacs[
    "image_path"
] = (
    standardized_eye_pacs[
        "image_id"
    ]
    .map(
        lambda image_id: str(
            EYEPACS_IMAGE_ROOT
            / f"{image_id}.jpeg"
        )
    )
)


assert standardized_eye_pacs[
    "dr_grade"
].between(
    0,
    4,
).all()


# ============================================================
# 5. Check patient-eye consistency
# ============================================================

eye_grade_nunique = (
    standardized_eye_pacs
    .groupby(
        [
            "patient_id",
            "eye_side",
        ]
    )[
        "dr_grade"
    ]
    .nunique()
    .rename(
        "n_unique_grades"
    )
    .reset_index()
)


inconsistent_eye_units = (
    eye_grade_nunique.loc[
        eye_grade_nunique[
            "n_unique_grades"
        ] > 1
    ]
    .copy()
)


eye_level_df = (
    standardized_eye_pacs
    .sort_values(
        [
            "patient_id",
            "eye_side",
            "image_path",
        ]
    )
    .groupby(
        [
            "patient_id",
            "eye_side",
        ],
        as_index=False,
    )
    .first()
)


# ============================================================
# 6. Identify bilateral patients
# ============================================================

eye_count_per_patient = (
    eye_level_df
    .groupby(
        "patient_id"
    )[
        "eye_side"
    ]
    .nunique()
)


bilateral_patient_ids = (
    eye_count_per_patient.loc[
        eye_count_per_patient == 2
    ]
    .index
    .astype(str)
    .tolist()
)


bilateral_eye_df = (
    eye_level_df.loc[
        eye_level_df[
            "patient_id"
        ].isin(
            bilateral_patient_ids
        )
    ]
    .copy()
)


bilateral_wide = (
    bilateral_eye_df
    .pivot(
        index="patient_id",
        columns="eye_side",
        values="dr_grade",
    )
    .reset_index()
)


assert "left" in bilateral_wide.columns
assert "right" in bilateral_wide.columns


bilateral_wide[
    "absolute_grade_gap"
] = (
    bilateral_wide["left"]
    - bilateral_wide["right"]
).abs()


unequal_grade_df = (
    bilateral_wide.loc[
        bilateral_wide[
            "absolute_grade_gap"
        ] > 0
    ]
    .copy()
)


unequal_grade_df[
    "left_higher_grade"
] = (
    unequal_grade_df["left"]
    > unequal_grade_df["right"]
).astype(int)


# ============================================================
# 7. Summary distributions
# ============================================================

audit_summary = pd.DataFrame(
    [
        {
            "metadata_rows": int(
                len(
                    standardized_eye_pacs
                )
            ),
            "eye_rows": int(
                len(
                    eye_level_df
                )
            ),
            "unique_patients": int(
                eye_level_df[
                    "patient_id"
                ].nunique()
            ),
            "bilateral_patients": int(
                len(
                    bilateral_wide
                )
            ),
            "unequal_grade_patients": int(
                len(
                    unequal_grade_df
                )
            ),
            "inconsistent_patient_eye_units": int(
                len(
                    inconsistent_eye_units
                )
            ),
        }
    ]
)


grade_gap_distribution = (
    unequal_grade_df[
        "absolute_grade_gap"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "absolute_grade_gap"
    )
    .reset_index(
        name="patients"
    )
)


direction_distribution = (
    unequal_grade_df[
        "left_higher_grade"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "left_higher_grade"
    )
    .reset_index(
        name="patients"
    )
)


number_of_unequal_patients = int(
    len(
        unequal_grade_df
    )
)


planned_held_out_patients = int(
    np.floor(
        0.25
        * number_of_unequal_patients
    )
)


both_directions_present = bool(
    unequal_grade_df[
        "left_higher_grade"
    ].nunique()
    == 2
)


passes_feasibility_gate = bool(
    number_of_unequal_patients
    >= MINIMUM_UNEQUAL_GRADE_PATIENTS
    and planned_held_out_patients
    >= MINIMUM_HELD_OUT_PATIENTS
    and both_directions_present
    and len(
        inconsistent_eye_units
    )
    == 0
)


# ============================================================
# 8. Permanent output paths in Google Drive
# ============================================================

CANONICAL_METADATA_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Canonical_Metadata_v0.1.csv"
)

AUDIT_SUMMARY_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Bilateral_Audit_Summary_v0.1.csv"
)

BILATERAL_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Bilateral_Patients_v0.1.csv"
)

UNEQUAL_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Unequal_Grade_Patients_v0.1.csv"
)

GRADE_GAP_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Grade_Gap_Distribution_v0.1.csv"
)

DIRECTION_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Direction_Distribution_v0.1.csv"
)

FEASIBILITY_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Feasibility_Decision_v0.1.json"
)


standardized_eye_pacs.to_csv(
    CANONICAL_METADATA_PATH,
    index=False,
)

audit_summary.to_csv(
    AUDIT_SUMMARY_PATH,
    index=False,
)

bilateral_wide.to_csv(
    BILATERAL_PATH,
    index=False,
)

unequal_grade_df.to_csv(
    UNEQUAL_PATH,
    index=False,
)

grade_gap_distribution.to_csv(
    GRADE_GAP_PATH,
    index=False,
)

direction_distribution.to_csv(
    DIRECTION_PATH,
    index=False,
)


feasibility_decision = {
    "dataset": (
        EYEPACS_DATASET_NAME
    ),
    "metadata_rows": int(
        len(
            standardized_eye_pacs
        )
    ),
    "bilateral_patients": int(
        len(
            bilateral_wide
        )
    ),
    "unequal_grade_patients": (
        number_of_unequal_patients
    ),
    "planned_held_out_patients": (
        planned_held_out_patients
    ),
    "both_directions_present": (
        both_directions_present
    ),
    "inconsistent_patient_eye_units": int(
        len(
            inconsistent_eye_units
        )
    ),
    "passes_feasibility_gate": (
        passes_feasibility_gate
    ),
    "decision": (
        "PASS_EYEPACS_METADATA_ADVANCE_TO_IMAGE_ACQUISITION"
        if passes_feasibility_gate
        else
        "FAIL_EYEPACS_METADATA_FEASIBILITY_GATE"
    ),
}


with open(
    FEASIBILITY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        feasibility_decision,
        file,
        indent=2,
    )


# ============================================================
# 9. Update candidate registry when available
# ============================================================

if CANDIDATE_REGISTRY_PATH.is_file():

    candidate_registry_df = pd.read_csv(
        CANDIDATE_REGISTRY_PATH
    )

    dataset_mask = (
        candidate_registry_df[
            "dataset_name"
        ].astype(str)
        == EYEPACS_DATASET_NAME
    )

    if dataset_mask.any():

        candidate_registry_df.loc[
            dataset_mask,
            "metadata_path",
        ] = str(
            PLANNED_METADATA_PATH
        )

        candidate_registry_df.loc[
            dataset_mask,
            "has_bilateral_patients",
        ] = bool(
            len(
                bilateral_wide
            ) > 0
        )

        candidate_registry_df.loc[
            dataset_mask,
            "supports_unequal_grade_pairs",
        ] = bool(
            number_of_unequal_patients > 0
        )

        candidate_registry_df.loc[
            dataset_mask,
            "notes",
        ] = (
            "Anonymous KaggleHub metadata download. "
            f"{len(bilateral_wide)} bilateral patients; "
            f"{number_of_unequal_patients} unequal-grade "
            f"patients; pass={passes_feasibility_gate}."
        )

        candidate_registry_df.to_csv(
            CANDIDATE_REGISTRY_PATH,
            index=False,
        )


# ============================================================
# 10. Final report
# ============================================================

print(
    "\n================ EYEPACS METADATA "
    "FEASIBILITY AUDIT ================"
)

display(
    audit_summary
)


print(
    "\nGrade-gap distribution:"
)

display(
    grade_gap_distribution
)


print(
    "\nDirection distribution:"
)

print(
    "0 = right eye has higher grade"
)

print(
    "1 = left eye has higher grade"
)

display(
    direction_distribution
)


print(
    "\nFeasibility decision:"
)

print(
    feasibility_decision[
        "decision"
    ]
)


print(
    "\nUnequal-grade patients:",
    number_of_unequal_patients,
)

print(
    "Planned held-out patients:",
    planned_held_out_patients,
)

print(
    "Both directions present:",
    both_directions_present,
)


print(
    "\nPermanent metadata file:"
)

print(
    PLANNED_METADATA_PATH
)


print(
    "\nEyePACS anonymous metadata audit completed."
)

================ EYEPACS ANONYMOUS METADATA SETUP ================
Drive root:
/content/drive/MyDrive

Permanent metadata path:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/EyePACS_2015_External_Replication/metadata/trainLabels.csv

Installing KaggleHub...

Attempting anonymous single-file download...


100%|██████████| 454k/454k [00:00<00:00, 9.84MB/s]


KaggleHub returned:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/EyePACS_2015_External_Replication/metadata/trainLabels.csv

Metadata permanently saved to Drive:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/EyePACS_2015_External_Replication/metadata/trainLabels.csv

================ RAW METADATA AUDIT ================
Raw metadata shape: (35126, 2)
Raw columns: ['image', 'level']



Unparsed filename rows: 0

================ EYEPACS METADATA FEASIBILITY AUDIT ================


,metadata_rows,eye_rows,unique_patients,bilateral_patients,unequal_grade_patients,inconsistent_patient_eye_units
0,35126,35126,17563,17563,2240,0



Grade-gap distribution:


,absolute_grade_gap,patients
0,1,1485
1,2,723
2,3,11
3,4,21



Direction distribution:
0 = right eye has higher grade
1 = left eye has higher grade


,left_higher_grade,patients
0,0,1066
1,1,1174



Feasibility decision:
PASS_EYEPACS_METADATA_ADVANCE_TO_IMAGE_ACQUISITION

Unequal-grade patients: 2240
Planned held-out patients: 560
Both directions present: True

Permanent metadata file:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/EyePACS_2015_External_Replication/metadata/trainLabels.csv

EyePACS anonymous metadata audit completed.


In [9]:
#@title 06. Freeze EyePACS patient split and image acquisition manifest

from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit

import json
import numpy as np
import pandas as pd


# ============================================================
# 1. Resolve Drive and paths
# ============================================================

if Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

elif Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")

else:
    raise FileNotFoundError(
        "Google Drive is not mounted. "
        "Please rerun the first notebook cell."
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

EXTERNAL_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "External_Replication_Protocol_v0.1"
)

EYEPACS_DATASET_ROOT = (
    PROJECT_ROOT
    / "02_Dataset_Map"
    / "EyePACS_2015_External_Replication"
)

EYEPACS_IMAGE_ROOT = (
    EYEPACS_DATASET_ROOT
    / "resized_train_cropped"
)

EYEPACS_IMAGE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


CANONICAL_METADATA_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Canonical_Metadata_v0.1.csv"
)

UNEQUAL_PATIENTS_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Unequal_Grade_Patients_v0.1.csv"
)

FEASIBILITY_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Feasibility_Decision_v0.1.json"
)


required_paths = [
    CANONICAL_METADATA_PATH,
    UNEQUAL_PATIENTS_PATH,
    FEASIBILITY_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing required metadata outputs:\n"
        + "\n".join(
            str(path)
            for path in missing_paths
        )
    )


# ============================================================
# 2. Verify feasibility gate
# ============================================================

with open(
    FEASIBILITY_PATH,
    "r",
    encoding="utf-8",
) as file:
    feasibility_decision = json.load(file)


assert (
    feasibility_decision["decision"]
    ==
    "PASS_EYEPACS_METADATA_ADVANCE_TO_IMAGE_ACQUISITION"
), feasibility_decision


assert int(
    feasibility_decision[
        "unequal_grade_patients"
    ]
) == 2240


assert int(
    feasibility_decision[
        "planned_held_out_patients"
    ]
) == 560


print(
    "================ FEASIBILITY IMPORT "
    "================"
)

print(
    "Decision:",
    feasibility_decision["decision"],
)

print(
    "Unequal-grade patients:",
    feasibility_decision[
        "unequal_grade_patients"
    ],
)

print(
    "Planned held-out patients:",
    feasibility_decision[
        "planned_held_out_patients"
    ],
)


# ============================================================
# 3. Load canonical metadata and patient pairs
# ============================================================

canonical_metadata = pd.read_csv(
    CANONICAL_METADATA_PATH
)

unequal_patients = pd.read_csv(
    UNEQUAL_PATIENTS_PATH
)


canonical_metadata[
    "patient_id"
] = (
    canonical_metadata[
        "patient_id"
    ].astype(str)
)

unequal_patients[
    "patient_id"
] = (
    unequal_patients[
        "patient_id"
    ].astype(str)
)


required_canonical_columns = [
    "patient_id",
    "eye_side",
    "dr_grade",
    "image_id",
    "image_path",
]

missing_canonical_columns = [
    column
    for column in required_canonical_columns
    if column not in canonical_metadata.columns
]

assert not missing_canonical_columns, (
    missing_canonical_columns
)


required_pair_columns = [
    "patient_id",
    "left",
    "right",
    "absolute_grade_gap",
    "left_higher_grade",
]

missing_pair_columns = [
    column
    for column in required_pair_columns
    if column not in unequal_patients.columns
]

assert not missing_pair_columns, (
    missing_pair_columns
)


assert len(unequal_patients) == 2240
assert unequal_patients["patient_id"].is_unique


# ============================================================
# 4. Resolve exact left- and right-eye image records
# ============================================================

left_metadata = (
    canonical_metadata.loc[
        canonical_metadata[
            "eye_side"
        ].eq("left"),
        [
            "patient_id",
            "image_id",
            "dr_grade",
        ],
    ]
    .rename(
        columns={
            "image_id": "left_image_id",
            "dr_grade": "left_image_grade",
        }
    )
    .copy()
)


right_metadata = (
    canonical_metadata.loc[
        canonical_metadata[
            "eye_side"
        ].eq("right"),
        [
            "patient_id",
            "image_id",
            "dr_grade",
        ],
    ]
    .rename(
        columns={
            "image_id": "right_image_id",
            "dr_grade": "right_image_grade",
        }
    )
    .copy()
)


assert left_metadata["patient_id"].is_unique
assert right_metadata["patient_id"].is_unique


pair_table = (
    unequal_patients
    .merge(
        left_metadata,
        on="patient_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        right_metadata,
        on="patient_id",
        how="left",
        validate="one_to_one",
    )
)


missing_image_id_mask = (
    pair_table[
        [
            "left_image_id",
            "right_image_id",
        ]
    ]
    .isna()
    .any(axis=1)
)

assert not missing_image_id_mask.any()


assert np.array_equal(
    pair_table["left"].astype(int).to_numpy(),
    pair_table[
        "left_image_grade"
    ].astype(int).to_numpy(),
)


assert np.array_equal(
    pair_table["right"].astype(int).to_numpy(),
    pair_table[
        "right_image_grade"
    ].astype(int).to_numpy(),
)


# ============================================================
# 5. Pre-register stratification variables
#
# Split preserves:
# - left-higher versus right-higher direction
# - grade gap 1 versus grade gap >= 2
# ============================================================

pair_table[
    "grade_gap_group"
] = np.where(
    pair_table[
        "absolute_grade_gap"
    ].astype(int)
    == 1,
    "gap_1",
    "gap_2_or_more",
)


pair_table[
    "split_stratum"
] = (
    "direction_"
    + pair_table[
        "left_higher_grade"
    ].astype(int).astype(str)
    + "__"
    + pair_table[
        "grade_gap_group"
    ].astype(str)
)


print(
    "\n================ PRE-SPLIT STRATA "
    "================"
)

pre_split_strata = (
    pair_table[
        "split_stratum"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "split_stratum"
    )
    .reset_index(
        name="patients"
    )
)

display(pre_split_strata)


# ============================================================
# 6. Freeze patient-level development/validation split
# ============================================================

RANDOM_SEED = 20260720
VALIDATION_PATIENTS = 560


splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=VALIDATION_PATIENTS,
    random_state=RANDOM_SEED,
)


development_indices, validation_indices = next(
    splitter.split(
        X=np.zeros(
            len(pair_table)
        ),
        y=pair_table[
            "split_stratum"
        ],
    )
)


pair_table[
    "split"
] = "development"

pair_table.loc[
    validation_indices,
    "split",
] = "validation"


assert (
    pair_table[
        "split"
    ].value_counts().to_dict()
    ==
    {
        "development": 1680,
        "validation": 560,
    }
)


assert (
    pair_table.loc[
        pair_table[
            "split"
        ].eq("development"),
        "patient_id",
    ]
    .isin(
        pair_table.loc[
            pair_table[
                "split"
            ].eq("validation"),
            "patient_id",
        ]
    )
    .sum()
    == 0
)


# ============================================================
# 7. Build two-row-per-patient image acquisition manifest
# ============================================================

left_manifest = pd.DataFrame(
    {
        "patient_id": (
            pair_table[
                "patient_id"
            ].astype(str)
        ),
        "split": pair_table["split"],
        "eye_side": "left",
        "image_id": (
            pair_table[
                "left_image_id"
            ].astype(str)
        ),
        "eye_grade": (
            pair_table[
                "left"
            ].astype(int)
        ),
        "fellow_eye_grade": (
            pair_table[
                "right"
            ].astype(int)
        ),
        "left_higher_grade": (
            pair_table[
                "left_higher_grade"
            ].astype(int)
        ),
        "absolute_grade_gap": (
            pair_table[
                "absolute_grade_gap"
            ].astype(int)
        ),
        "grade_gap_group": (
            pair_table[
                "grade_gap_group"
            ]
        ),
        "split_stratum": (
            pair_table[
                "split_stratum"
            ]
        ),
    }
)


right_manifest = pd.DataFrame(
    {
        "patient_id": (
            pair_table[
                "patient_id"
            ].astype(str)
        ),
        "split": pair_table["split"],
        "eye_side": "right",
        "image_id": (
            pair_table[
                "right_image_id"
            ].astype(str)
        ),
        "eye_grade": (
            pair_table[
                "right"
            ].astype(int)
        ),
        "fellow_eye_grade": (
            pair_table[
                "left"
            ].astype(int)
        ),
        "left_higher_grade": (
            pair_table[
                "left_higher_grade"
            ].astype(int)
        ),
        "absolute_grade_gap": (
            pair_table[
                "absolute_grade_gap"
            ].astype(int)
        ),
        "grade_gap_group": (
            pair_table[
                "grade_gap_group"
            ]
        ),
        "split_stratum": (
            pair_table[
                "split_stratum"
            ]
        ),
    }
)


image_manifest = pd.concat(
    [
        left_manifest,
        right_manifest,
    ],
    ignore_index=True,
)


image_manifest = (
    image_manifest
    .sort_values(
        [
            "split",
            "patient_id",
            "eye_side",
        ]
    )
    .reset_index(drop=True)
)


image_manifest[
    "kaggle_relative_path"
] = (
    "resized_train_cropped/"
    + image_manifest[
        "image_id"
    ].astype(str)
    + ".jpeg"
)


image_manifest[
    "local_image_path"
] = (
    image_manifest[
        "image_id"
    ]
    .map(
        lambda image_id: str(
            EYEPACS_IMAGE_ROOT
            / f"{image_id}.jpeg"
        )
    )
)


image_manifest[
    "image_exists"
] = (
    image_manifest[
        "local_image_path"
    ]
    .map(
        lambda path: Path(
            path
        ).is_file()
    )
)


image_manifest[
    "manifest_row"
] = np.arange(
    len(image_manifest)
)


# ============================================================
# 8. Integrity checks
# ============================================================

assert len(image_manifest) == 4480

assert (
    image_manifest[
        "patient_id"
    ].nunique()
    == 2240
)

assert (
    image_manifest
    .groupby(
        "patient_id"
    )
    .size()
    .eq(2)
    .all()
)

assert (
    image_manifest
    .groupby(
        "patient_id"
    )[
        "eye_side"
    ]
    .nunique()
    .eq(2)
    .all()
)

assert image_manifest[
    "image_id"
].is_unique


validation_manifest = (
    image_manifest.loc[
        image_manifest[
            "split"
        ].eq("validation")
    ]
)

assert len(validation_manifest) == 1120

assert (
    validation_manifest[
        "patient_id"
    ].nunique()
    == 560
)


# ============================================================
# 9. Split audit tables
# ============================================================

split_summary = (
    pair_table
    .groupby(
        "split"
    )
    .agg(
        patients=(
            "patient_id",
            "nunique",
        ),
        left_higher_patients=(
            "left_higher_grade",
            "sum",
        ),
        mean_absolute_grade_gap=(
            "absolute_grade_gap",
            "mean",
        ),
    )
    .reset_index()
)


split_summary[
    "right_higher_patients"
] = (
    split_summary[
        "patients"
    ]
    - split_summary[
        "left_higher_patients"
    ]
)


split_stratum_audit = (
    pair_table
    .groupby(
        [
            "split",
            "split_stratum",
        ]
    )
    .size()
    .rename(
        "patients"
    )
    .reset_index()
)


grade_gap_audit = (
    pair_table
    .groupby(
        [
            "split",
            "absolute_grade_gap",
        ]
    )
    .size()
    .rename(
        "patients"
    )
    .reset_index()
)


# ============================================================
# 10. Save sealed split and manifest
# ============================================================

PATIENT_SPLIT_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Patient_Split_v0.1.csv"
)

IMAGE_MANIFEST_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Image_Acquisition_Manifest_v0.1.csv"
)

SPLIT_SUMMARY_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Split_Summary_v0.1.csv"
)

SPLIT_STRATUM_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Split_Stratum_Audit_v0.1.csv"
)

GRADE_GAP_AUDIT_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Split_Grade_Gap_Audit_v0.1.csv"
)

SPLIT_DECISION_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Split_Decision_v0.1.json"
)


pair_table.to_csv(
    PATIENT_SPLIT_PATH,
    index=False,
)

image_manifest.to_csv(
    IMAGE_MANIFEST_PATH,
    index=False,
)

split_summary.to_csv(
    SPLIT_SUMMARY_PATH,
    index=False,
)

split_stratum_audit.to_csv(
    SPLIT_STRATUM_PATH,
    index=False,
)

grade_gap_audit.to_csv(
    GRADE_GAP_AUDIT_PATH,
    index=False,
)


split_decision = {
    "decision": (
        "PASS_EYEPACS_PATIENT_SPLIT_ADVANCE_TO_TARGETED_IMAGE_ACQUISITION"
    ),
    "random_seed": RANDOM_SEED,
    "total_unequal_grade_patients": 2240,
    "development_patients": 1680,
    "validation_patients": 560,
    "target_images": 4480,
    "patient_overlap": 0,
    "stratification": [
        "left_higher_grade",
        "grade_gap_1_versus_2_or_more",
    ],
    "validation_use_boundary": (
        "Validation labels and predictions must not be "
        "used for model selection or condition selection."
    ),
}


with open(
    SPLIT_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        split_decision,
        file,
        indent=2,
    )


# ============================================================
# 11. Final report
# ============================================================

print(
    "\n================ EYEPACS SEALED "
    "PATIENT SPLIT ================"
)

display(split_summary)


print(
    "\nSplit-stratum audit:"
)

display(split_stratum_audit)


print(
    "\nGrade-gap audit:"
)

display(grade_gap_audit)


print(
    "\nImages currently present:",
    int(
        image_manifest[
            "image_exists"
        ].sum()
    ),
    "/",
    len(image_manifest),
)


print(
    "\nDecision:"
)

print(
    split_decision[
        "decision"
    ]
)


print("\nSaved:")
print(PATIENT_SPLIT_PATH)
print(IMAGE_MANIFEST_PATH)
print(SPLIT_SUMMARY_PATH)
print(SPLIT_DECISION_PATH)


print(
    "\nEyePACS patient split and image "
    "acquisition manifest sealed."
)

================ FEASIBILITY IMPORT ================
Decision: PASS_EYEPACS_METADATA_ADVANCE_TO_IMAGE_ACQUISITION
Unequal-grade patients: 2240
Planned held-out patients: 560

================ PRE-SPLIT STRATA ================


,split_stratum,patients
0,direction_0__gap_1,701
1,direction_0__gap_2_or_more,365
2,direction_1__gap_1,784
3,direction_1__gap_2_or_more,390



================ EYEPACS SEALED PATIENT SPLIT ================


,split,patients,left_higher_patients,mean_absolute_grade_gap,right_higher_patients
0,development,1680,880,1.357738,800
1,validation,560,294,1.369643,266



Split-stratum audit:


,split,split_stratum,patients
0,development,direction_0__gap_1,526
1,development,direction_0__gap_2_or_more,274
2,development,direction_1__gap_1,588
3,development,direction_1__gap_2_or_more,292
4,validation,direction_0__gap_1,175
5,validation,direction_0__gap_2_or_more,91
6,validation,direction_1__gap_1,196
7,validation,direction_1__gap_2_or_more,98



Grade-gap audit:


,split,absolute_grade_gap,patients
0,development,1,1114
1,development,2,544
2,development,3,9
3,development,4,13
4,validation,1,371
5,validation,2,179
6,validation,3,2
7,validation,4,8



Images currently present: 0 / 4480

Decision:
PASS_EYEPACS_PATIENT_SPLIT_ADVANCE_TO_TARGETED_IMAGE_ACQUISITION

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/EyePACS_2015_Patient_Split_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/EyePACS_2015_Image_Acquisition_Manifest_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/EyePACS_2015_Split_Summary_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/EyePACS_2015_Split_Decision_v0.1.json

EyePACS patient split and image acquisition manifest sealed.


In [10]:
#@title 07. Audit availability of the sealed images in the cropped EyePACS subset

from pathlib import Path

import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd


# ============================================================
# 1. Resolve Drive and paths
# ============================================================

if Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

elif Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")

else:
    raise FileNotFoundError(
        "Google Drive is not mounted. "
        "Please rerun the first notebook cell."
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

EXTERNAL_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "External_Replication_Protocol_v0.1"
)

EYEPACS_ROOT = (
    PROJECT_ROOT
    / "02_Dataset_Map"
    / "EyePACS_2015_External_Replication"
)

METADATA_ROOT = (
    EYEPACS_ROOT
    / "metadata"
)

IMAGE_ROOT = (
    EYEPACS_ROOT
    / "resized_train_cropped"
)

METADATA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

IMAGE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


DATASET_HANDLE = (
    "tanlikesmath/diabetic-retinopathy-resized"
)

ORIGINAL_MANIFEST_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Image_Acquisition_Manifest_v0.1.csv"
)

PATIENT_SPLIT_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Patient_Split_v0.1.csv"
)

SPLIT_DECISION_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Split_Decision_v0.1.json"
)

CROPPED_LABEL_PATH = (
    METADATA_ROOT
    / "trainLabels_cropped.csv"
)


required_paths = [
    ORIGINAL_MANIFEST_PATH,
    PATIENT_SPLIT_PATH,
    SPLIT_DECISION_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing sealed split artifacts:\n"
        + "\n".join(
            str(path)
            for path in missing_paths
        )
    )


# ============================================================
# 2. Verify the sealed split
# ============================================================

with open(
    SPLIT_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    split_decision = json.load(file)


assert (
    split_decision["decision"]
    ==
    "PASS_EYEPACS_PATIENT_SPLIT_ADVANCE_TO_TARGETED_IMAGE_ACQUISITION"
), split_decision


original_manifest = pd.read_csv(
    ORIGINAL_MANIFEST_PATH
)

patient_split = pd.read_csv(
    PATIENT_SPLIT_PATH
)


original_manifest["patient_id"] = (
    original_manifest[
        "patient_id"
    ].astype(str)
)

original_manifest["image_id"] = (
    original_manifest[
        "image_id"
    ].astype(str)
)

patient_split["patient_id"] = (
    patient_split[
        "patient_id"
    ].astype(str)
)


assert len(original_manifest) == 4480

assert (
    original_manifest[
        "patient_id"
    ].nunique()
    == 2240
)

assert (
    original_manifest
    .groupby("patient_id")
    .size()
    .eq(2)
    .all()
)

assert (
    original_manifest[
        "image_id"
    ].is_unique
)


print(
    "================ SEALED MANIFEST IMPORT "
    "================"
)

print(
    "Target patients:",
    original_manifest[
        "patient_id"
    ].nunique(),
)

print(
    "Target images:",
    len(original_manifest),
)


# ============================================================
# 3. Pre-register availability rules before reading the file
# ============================================================

MINIMUM_DEVELOPMENT_COMPLETE_PAIRS = 1500
MINIMUM_VALIDATION_COMPLETE_PAIRS = 500
MAXIMUM_INCOMPLETE_PAIR_FRACTION = 0.02


availability_protocol = {
    "rule_1": (
        "Retain only patients for whom both sealed left- "
        "and right-eye image IDs occur in trainLabels_cropped.csv."
    ),
    "rule_2": (
        "Do not transfer patients between development "
        "and validation."
    ),
    "rule_3": (
        "Do not replace unavailable patients using their "
        "labels or model results."
    ),
    "minimum_development_complete_pairs": (
        MINIMUM_DEVELOPMENT_COMPLETE_PAIRS
    ),
    "minimum_validation_complete_pairs": (
        MINIMUM_VALIDATION_COMPLETE_PAIRS
    ),
    "maximum_incomplete_pair_fraction": (
        MAXIMUM_INCOMPLETE_PAIR_FRACTION
    ),
}


# ============================================================
# 4. Download cropped metadata only
# ============================================================

if CROPPED_LABEL_PATH.is_file():

    print(
        "\nExisting trainLabels_cropped.csv detected."
    )

    print(
        "Skipping download:"
    )

    print(CROPPED_LABEL_PATH)

else:

    print(
        "\nInstalling or updating KaggleHub..."
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "kagglehub",
        ],
        check=True,
    )

    import kagglehub

    print(
        "\nDownloading trainLabels_cropped.csv only..."
    )

    downloaded_result = (
        kagglehub.dataset_download(
            DATASET_HANDLE,
            path="trainLabels_cropped.csv",
            output_dir=str(
                METADATA_ROOT
            ),
            force_download=True,
        )
    )

    returned_path = Path(
        str(downloaded_result)
    )


    candidates = list(
        METADATA_ROOT.rglob(
            "trainLabels_cropped.csv"
        )
    )


    if (
        not candidates
        and returned_path.is_file()
        and returned_path.name
        == "trainLabels_cropped.csv"
    ):
        candidates = [
            returned_path
        ]


    if (
        not candidates
        and returned_path.is_dir()
    ):
        candidates = list(
            returned_path.rglob(
                "trainLabels_cropped.csv"
            )
        )


    if not candidates:
        raise FileNotFoundError(
            "trainLabels_cropped.csv could not "
            "be located after download."
        )


    source_path = candidates[0]


    if (
        source_path.resolve()
        != CROPPED_LABEL_PATH.resolve()
    ):
        shutil.copy2(
            source_path,
            CROPPED_LABEL_PATH,
        )


assert CROPPED_LABEL_PATH.is_file()


print(
    "\nCropped metadata ready:"
)

print(CROPPED_LABEL_PATH)


# ============================================================
# 5. Load and standardise cropped metadata
# ============================================================

cropped_labels = pd.read_csv(
    CROPPED_LABEL_PATH
)


# Remove accidental saved index columns if present.
unnamed_columns = [
    column
    for column in cropped_labels.columns
    if str(column).startswith("Unnamed:")
]

if unnamed_columns:
    cropped_labels = cropped_labels.drop(
        columns=unnamed_columns
    )


print(
    "\n================ CROPPED METADATA AUDIT "
    "================"
)

print(
    "Shape:",
    cropped_labels.shape,
)

print(
    "Columns:",
    cropped_labels.columns.tolist(),
)


required_columns = [
    "image",
    "level",
]

missing_columns = [
    column
    for column in required_columns
    if column not in cropped_labels.columns
]

if missing_columns:
    raise ValueError(
        "Missing required cropped-label columns: "
        f"{missing_columns}"
    )


cropped_labels[
    "image_id"
] = (
    cropped_labels["image"]
    .astype(str)
    .map(
        lambda value: Path(value).stem
    )
)


cropped_labels[
    "dr_grade_cropped"
] = pd.to_numeric(
    cropped_labels["level"],
    errors="raise",
).astype(int)


assert cropped_labels[
    "image_id"
].is_unique

assert cropped_labels[
    "dr_grade_cropped"
].between(
    0,
    4,
).all()


cropped_grade_lookup = dict(
    zip(
        cropped_labels[
            "image_id"
        ],
        cropped_labels[
            "dr_grade_cropped"
        ],
    )
)


cropped_image_ids = set(
    cropped_labels[
        "image_id"
    ]
)


# ============================================================
# 6. Compare sealed image IDs with cropped availability
# ============================================================

availability_manifest = (
    original_manifest.copy()
)


availability_manifest[
    "cropped_metadata_available"
] = (
    availability_manifest[
        "image_id"
    ].isin(
        cropped_image_ids
    )
)


availability_manifest[
    "cropped_metadata_grade"
] = (
    availability_manifest[
        "image_id"
    ].map(
        cropped_grade_lookup
    )
)


available_rows = (
    availability_manifest[
        "cropped_metadata_available"
    ]
)


grade_mismatch_mask = (
    availability_manifest.loc[
        available_rows,
        "eye_grade",
    ].astype(int).to_numpy()
    !=
    availability_manifest.loc[
        available_rows,
        "cropped_metadata_grade",
    ].astype(int).to_numpy()
)


number_of_grade_mismatches = int(
    grade_mismatch_mask.sum()
)


assert number_of_grade_mismatches == 0, (
    "The cropped-label grade does not match "
    "the sealed trainLabels.csv grade."
)


# ============================================================
# 7. Patient-pair availability
# ============================================================

pair_availability = (
    availability_manifest
    .groupby(
        [
            "patient_id",
            "split",
        ],
        as_index=False,
    )
    .agg(
        target_images=(
            "image_id",
            "size",
        ),
        available_images=(
            "cropped_metadata_available",
            "sum",
        ),
    )
)


pair_availability[
    "complete_pair"
] = (
    pair_availability[
        "available_images"
    ]
    == 2
)


pair_availability[
    "missing_images"
] = (
    pair_availability[
        "target_images"
    ]
    - pair_availability[
        "available_images"
    ]
)


assert (
    pair_availability[
        "target_images"
    ]
    == 2
).all()


complete_patient_ids = set(
    pair_availability.loc[
        pair_availability[
            "complete_pair"
        ],
        "patient_id",
    ]
)


filtered_manifest = (
    availability_manifest.loc[
        availability_manifest[
            "patient_id"
        ].isin(
            complete_patient_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


filtered_manifest[
    "acquisition_row"
] = np.arange(
    len(filtered_manifest)
)


assert (
    filtered_manifest
    .groupby("patient_id")
    .size()
    .eq(2)
    .all()
)


# ============================================================
# 8. Availability summary
# ============================================================

availability_summary = (
    pair_availability
    .groupby(
        "split"
    )
    .agg(
        target_patients=(
            "patient_id",
            "nunique",
        ),
        complete_pairs=(
            "complete_pair",
            "sum",
        ),
        target_images=(
            "target_images",
            "sum",
        ),
        available_images=(
            "available_images",
            "sum",
        ),
        missing_images=(
            "missing_images",
            "sum",
        ),
    )
    .reset_index()
)


availability_summary[
    "incomplete_pairs"
] = (
    availability_summary[
        "target_patients"
    ]
    - availability_summary[
        "complete_pairs"
    ]
)


availability_summary[
    "incomplete_pair_fraction"
] = (
    availability_summary[
        "incomplete_pairs"
    ]
    / availability_summary[
        "target_patients"
    ]
)


development_complete_pairs = int(
    availability_summary.loc[
        availability_summary[
            "split"
        ].eq("development"),
        "complete_pairs",
    ].iloc[0]
)


validation_complete_pairs = int(
    availability_summary.loc[
        availability_summary[
            "split"
        ].eq("validation"),
        "complete_pairs",
    ].iloc[0]
)


total_target_pairs = int(
    pair_availability.shape[0]
)


total_complete_pairs = int(
    pair_availability[
        "complete_pair"
    ].sum()
)


total_incomplete_pairs = (
    total_target_pairs
    - total_complete_pairs
)


incomplete_pair_fraction = (
    total_incomplete_pairs
    / total_target_pairs
)


passes_availability_gate = bool(
    development_complete_pairs
    >= MINIMUM_DEVELOPMENT_COMPLETE_PAIRS
    and validation_complete_pairs
    >= MINIMUM_VALIDATION_COMPLETE_PAIRS
    and incomplete_pair_fraction
    <= MAXIMUM_INCOMPLETE_PAIR_FRACTION
    and number_of_grade_mismatches
    == 0
)


# ============================================================
# 9. Missing-image audit
# ============================================================

missing_target_images = (
    availability_manifest.loc[
        ~availability_manifest[
            "cropped_metadata_available"
        ]
    ]
    .copy()
)


incomplete_pairs = (
    pair_availability.loc[
        ~pair_availability[
            "complete_pair"
        ]
    ]
    .copy()
)


# ============================================================
# 10. Save outputs
# ============================================================

AVAILABILITY_PROTOCOL_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Cropped_Availability_Protocol_v0.1.json"
)

AVAILABILITY_MANIFEST_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Cropped_Availability_Manifest_v0.1.csv"
)

FILTERED_ACQUISITION_MANIFEST_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Filtered_Image_Acquisition_Manifest_v0.1.csv"
)

PAIR_AVAILABILITY_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Cropped_Pair_Availability_v0.1.csv"
)

AVAILABILITY_SUMMARY_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Cropped_Availability_Summary_v0.1.csv"
)

MISSING_IMAGES_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Missing_Cropped_Target_Images_v0.1.csv"
)

INCOMPLETE_PAIRS_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Incomplete_Cropped_Pairs_v0.1.csv"
)

AVAILABILITY_DECISION_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Cropped_Availability_Decision_v0.1.json"
)


with open(
    AVAILABILITY_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        availability_protocol,
        file,
        indent=2,
    )


availability_manifest.to_csv(
    AVAILABILITY_MANIFEST_PATH,
    index=False,
)

filtered_manifest.to_csv(
    FILTERED_ACQUISITION_MANIFEST_PATH,
    index=False,
)

pair_availability.to_csv(
    PAIR_AVAILABILITY_PATH,
    index=False,
)

availability_summary.to_csv(
    AVAILABILITY_SUMMARY_PATH,
    index=False,
)

missing_target_images.to_csv(
    MISSING_IMAGES_PATH,
    index=False,
)

incomplete_pairs.to_csv(
    INCOMPLETE_PAIRS_PATH,
    index=False,
)


availability_decision = {
    "decision": (
        "PASS_EYEPACS_CROPPED_AVAILABILITY_ADVANCE_TO_IMAGE_DOWNLOAD"
        if passes_availability_gate
        else
        "FAIL_EYEPACS_CROPPED_AVAILABILITY_GATE"
    ),
    "original_target_patients": (
        total_target_pairs
    ),
    "complete_cropped_pairs": (
        total_complete_pairs
    ),
    "incomplete_cropped_pairs": (
        total_incomplete_pairs
    ),
    "incomplete_pair_fraction": float(
        incomplete_pair_fraction
    ),
    "development_complete_pairs": (
        development_complete_pairs
    ),
    "validation_complete_pairs": (
        validation_complete_pairs
    ),
    "filtered_target_images": int(
        len(filtered_manifest)
    ),
    "grade_mismatches": (
        number_of_grade_mismatches
    ),
    "patient_split_changed": False,
    "patients_moved_between_splits": 0,
}


with open(
    AVAILABILITY_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        availability_decision,
        file,
        indent=2,
    )


# ============================================================
# 11. Final report
# ============================================================

print(
    "\n================ CROPPED IMAGE "
    "AVAILABILITY SUMMARY ================"
)

display(
    availability_summary
)


print(
    "\nTotal complete pairs:",
    total_complete_pairs,
    "/",
    total_target_pairs,
)

print(
    "Total incomplete pairs:",
    total_incomplete_pairs,
)

print(
    "Incomplete-pair fraction:",
    f"{incomplete_pair_fraction:.6f}",
)

print(
    "Grade mismatches:",
    number_of_grade_mismatches,
)

print(
    "Filtered acquisition images:",
    len(filtered_manifest),
)


print(
    "\nDecision:"
)

print(
    availability_decision[
        "decision"
    ]
)


if len(missing_target_images) > 0:

    print(
        "\nMissing target-image examples:"
    )

    display(
        missing_target_images[
            [
                "patient_id",
                "split",
                "eye_side",
                "image_id",
                "eye_grade",
            ]
        ].head(20)
    )


print("\nSaved:")
print(
    FILTERED_ACQUISITION_MANIFEST_PATH
)
print(
    AVAILABILITY_SUMMARY_PATH
)
print(
    AVAILABILITY_DECISION_PATH
)


print(
    "\nEyePACS cropped-image availability "
    "audit completed."
)

================ SEALED MANIFEST IMPORT ================
Target patients: 2240
Target images: 4480

Installing or updating KaggleHub...

Using Colab cache for faster access to the 'diabetic-retinopathy-resized' dataset.

Cropped metadata ready:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/EyePACS_2015_External_Replication/metadata/trainLabels_cropped.csv

================ CROPPED METADATA AUDIT ================
Shape: (35108, 2)
Columns: ['image', 'level']

================ CROPPED IMAGE AVAILABILITY SUMMARY ================


,split,target_patients,complete_pairs,target_images,available_images,missing_images,incomplete_pairs,incomplete_pair_fraction
0,development,1680,1675,3360,3354,6,5,0.002976
1,validation,560,559,1120,1119,1,1,0.001786



Total complete pairs: 2234 / 2240
Total incomplete pairs: 6
Incomplete-pair fraction: 0.002679
Grade mismatches: 0
Filtered acquisition images: 4468

Decision:
PASS_EYEPACS_CROPPED_AVAILABILITY_ADVANCE_TO_IMAGE_DOWNLOAD

Missing target-image examples:


,patient_id,split,eye_side,image_id,eye_grade
478,1557,development,left,1557_left,0
832,1986,development,left,1986_left,0
1004,21720,development,left,21720_left,1
2368,3829,development,left,3829_left,2
3310,9590,development,left,9590_left,2
3311,9590,development,right,9590_right,3
3392,10996,validation,left,10996_left,0



Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/EyePACS_2015_Filtered_Image_Acquisition_Manifest_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/EyePACS_2015_Cropped_Availability_Summary_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/EyePACS_2015_Cropped_Availability_Decision_v0.1.json

EyePACS cropped-image availability audit completed.


In [11]:
#@title 08. Acquire the 4,468 sealed EyePACS images through a temporary full-dataset download

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

import hashlib
import json
import os
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd


# ============================================================
# 1. Resolve Google Drive and project paths
# ============================================================

if Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

elif Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")

else:
    raise FileNotFoundError(
        "Google Drive is not mounted. "
        "Please rerun the first notebook cell."
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

EXTERNAL_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "External_Replication_Protocol_v0.1"
)

EYEPACS_DATASET_ROOT = (
    PROJECT_ROOT
    / "02_Dataset_Map"
    / "EyePACS_2015_External_Replication"
)

EYEPACS_IMAGE_ROOT = (
    EYEPACS_DATASET_ROOT
    / "resized_train_cropped"
)

EYEPACS_IMAGE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# Freeze the exact Kaggle dataset version.
DATASET_HANDLE = (
    "tanlikesmath/"
    "diabetic-retinopathy-resized/"
    "versions/7"
)

DATASET_VERSION = 7


FILTERED_MANIFEST_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Filtered_Image_Acquisition_Manifest_v0.1.csv"
)

AVAILABILITY_DECISION_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Cropped_Availability_Decision_v0.1.json"
)


required_paths = [
    FILTERED_MANIFEST_PATH,
    AVAILABILITY_DECISION_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing required availability artifacts:\n"
        + "\n".join(
            str(path)
            for path in missing_paths
        )
    )


# ============================================================
# 2. Verify the sealed availability decision
# ============================================================

with open(
    AVAILABILITY_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    availability_decision = json.load(file)


assert (
    availability_decision["decision"]
    ==
    "PASS_EYEPACS_CROPPED_AVAILABILITY_ADVANCE_TO_IMAGE_DOWNLOAD"
), availability_decision


manifest = pd.read_csv(
    FILTERED_MANIFEST_PATH
)


manifest["patient_id"] = (
    manifest["patient_id"]
    .astype(str)
)

manifest["image_id"] = (
    manifest["image_id"]
    .astype(str)
)


assert len(manifest) == 4468
assert manifest["patient_id"].nunique() == 2234
assert manifest["image_id"].is_unique

assert (
    manifest
    .groupby("patient_id")
    .size()
    .eq(2)
    .all()
)

assert (
    manifest
    .groupby("patient_id")["eye_side"]
    .nunique()
    .eq(2)
    .all()
)


manifest["local_image_path"] = (
    manifest["image_id"]
    .map(
        lambda image_id: str(
            EYEPACS_IMAGE_ROOT
            / f"{image_id}.jpeg"
        )
    )
)


print(
    "================ SEALED IMAGE "
    "MANIFEST IMPORT ================"
)

print(
    "Dataset version:",
    DATASET_VERSION,
)

print(
    "Patients:",
    manifest["patient_id"].nunique(),
)

print(
    "Images:",
    len(manifest),
)

print(
    "\nPermanent image root:"
)

print(EYEPACS_IMAGE_ROOT)


# ============================================================
# 3. Image inspection helper
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def inspect_image_file(
    path,
    compute_hash=False,
):
    path = Path(path)

    result = {
        "file_exists": path.is_file(),
        "file_valid": False,
        "file_size_bytes": np.nan,
        "image_width": np.nan,
        "image_height": np.nan,
        "image_mode": None,
        "sha256": None,
        "inspection_error": None,
    }

    if not path.is_file():
        return result

    try:
        file_size = int(
            path.stat().st_size
        )

        if file_size <= 0:
            raise ValueError(
                "File size is zero."
            )

        with Image.open(path) as image:
            image.load()

            width, height = image.size
            image_mode = image.mode

        if width <= 0 or height <= 0:
            raise ValueError(
                "Invalid image dimensions."
            )

        result[
            "file_valid"
        ] = True

        result[
            "file_size_bytes"
        ] = file_size

        result[
            "image_width"
        ] = int(width)

        result[
            "image_height"
        ] = int(height)

        result[
            "image_mode"
        ] = str(image_mode)

        if compute_hash:
            result[
                "sha256"
            ] = sha256_file(path)

    except Exception as error:
        result[
            "inspection_error"
        ] = repr(error)

    return result


# ============================================================
# 4. Audit any images already saved in Drive
#
# This makes the cell safely rerunnable after interruption.
# ============================================================

initial_audit_rows = []

for row in tqdm(
    manifest.itertuples(
        index=False
    ),
    total=len(manifest),
    desc="Checking existing Drive images",
):
    inspection = inspect_image_file(
        row.local_image_path,
        compute_hash=False,
    )

    initial_audit_rows.append(
        {
            "image_id": row.image_id,
            **inspection,
        }
    )


initial_audit = pd.DataFrame(
    initial_audit_rows
)


valid_existing_ids = set(
    initial_audit.loc[
        initial_audit[
            "file_valid"
        ],
        "image_id",
    ].astype(str)
)


required_image_ids = set(
    manifest["image_id"]
)


ids_requiring_copy = sorted(
    required_image_ids
    - valid_existing_ids
)


print(
    "\nValid images already present:",
    len(valid_existing_ids),
    "/",
    len(manifest),
)

print(
    "Images requiring acquisition:",
    len(ids_requiring_copy),
)


# ============================================================
# 5. Download the full dataset only when images are missing
#
# Full data remain on temporary Colab storage, not Drive.
# ============================================================

SCRATCH_ROOT = Path(
    "/content/eyepacs_v7_temporary_download"
)

SOURCE_CROPPED_ROOT = None

downloaded_this_run_ids = set()


if ids_requiring_copy:

    # --------------------------------------------------------
    # 5.1 Check temporary Colab storage
    # --------------------------------------------------------

    colab_disk = shutil.disk_usage(
        "/content"
    )

    free_gb = (
        colab_disk.free
        / (1024 ** 3)
    )

    print(
        "\nTemporary Colab free space:",
        f"{free_gb:.2f} GB",
    )

    MINIMUM_TEMPORARY_FREE_GB = 20.0

    if free_gb < MINIMUM_TEMPORARY_FREE_GB:
        raise RuntimeError(
            "Insufficient temporary Colab disk space.\n"
            f"Available: {free_gb:.2f} GB\n"
            f"Required: at least "
            f"{MINIMUM_TEMPORARY_FREE_GB:.1f} GB."
        )


    # --------------------------------------------------------
    # 5.2 Install KaggleHub
    # --------------------------------------------------------

    print(
        "\nInstalling or updating KaggleHub..."
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "kagglehub",
        ],
        check=True,
    )

    import kagglehub


    # --------------------------------------------------------
    # 5.3 Download Version 7 to temporary runtime storage
    # --------------------------------------------------------

    SCRATCH_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "\nDownloading the complete EyePACS "
        "resized dataset to temporary Colab storage."
    )

    print(
        "Nothing from the full archive is being "
        "stored permanently in Drive."
    )


    try:
        downloaded_result = (
            kagglehub.dataset_download(
                DATASET_HANDLE,
                output_dir=str(
                    SCRATCH_ROOT
                ),
                force_download=False,
            )
        )

    except Exception as error:
        raise RuntimeError(
            "The temporary full-dataset download failed. "
            "Rerun this cell; images already copied to "
            "Drive will be skipped."
        ) from error


    downloaded_root = Path(
        str(downloaded_result)
    )


    print(
        "\nKaggleHub returned:"
    )

    print(downloaded_root)


    # --------------------------------------------------------
    # 5.4 Locate resized_train_cropped
    # --------------------------------------------------------

    search_roots = [
        downloaded_root,
        SCRATCH_ROOT,
    ]

    cropped_directory_candidates = []

    for search_root in search_roots:

        if not search_root.exists():
            continue

        if (
            search_root.is_dir()
            and search_root.name
            == "resized_train_cropped"
        ):
            cropped_directory_candidates.append(
                search_root
            )

        if search_root.is_dir():
            cropped_directory_candidates.extend(
                [
                    path
                    for path in search_root.rglob(
                        "resized_train_cropped"
                    )
                    if path.is_dir()
                ]
            )


    # Remove duplicate paths.
    unique_candidates = []

    seen_candidate_paths = set()

    for candidate in cropped_directory_candidates:
        resolved_candidate = (
            candidate.resolve()
        )

        if resolved_candidate not in seen_candidate_paths:
            seen_candidate_paths.add(
                resolved_candidate
            )

            unique_candidates.append(
                candidate
            )


    if not unique_candidates:
        raise FileNotFoundError(
            "The resized_train_cropped directory "
            "was not found in the downloaded dataset."
        )


    def count_candidate_images(
        directory,
    ):
        return sum(
            1
            for path in directory.rglob("*")
            if (
                path.is_file()
                and path.suffix.lower()
                in {
                    ".jpeg",
                    ".jpg",
                    ".png",
                }
            )
        )


    candidate_counts = [
        (
            candidate,
            count_candidate_images(
                candidate
            ),
        )
        for candidate in unique_candidates
    ]


    SOURCE_CROPPED_ROOT, source_image_count = max(
        candidate_counts,
        key=lambda item: item[1],
    )


    print(
        "\nResolved cropped source directory:"
    )

    print(SOURCE_CROPPED_ROOT)

    print(
        "Images detected in source directory:",
        source_image_count,
    )


    # --------------------------------------------------------
    # 5.5 Build a source lookup for the sealed target IDs
    # --------------------------------------------------------

    target_ids_for_lookup = set(
        ids_requiring_copy
    )

    source_lookup = {}


    for source_path in tqdm(
        SOURCE_CROPPED_ROOT.rglob("*"),
        desc="Indexing cropped source files",
    ):
        if not source_path.is_file():
            continue

        if (
            source_path.suffix.lower()
            not in {
                ".jpeg",
                ".jpg",
                ".png",
            }
        ):
            continue

        image_id = source_path.stem

        if image_id in target_ids_for_lookup:
            source_lookup[
                image_id
            ] = source_path


    missing_source_ids = sorted(
        target_ids_for_lookup
        - set(
            source_lookup
        )
    )


    if missing_source_ids:
        raise FileNotFoundError(
            "Some sealed target images were not found "
            "inside the downloaded cropped directory.\n"
            f"Missing source images: "
            f"{len(missing_source_ids)}\n"
            f"Examples: "
            f"{missing_source_ids[:20]}"
        )


    # --------------------------------------------------------
    # 5.6 Estimate selected-image storage requirement
    # --------------------------------------------------------

    selected_source_bytes = int(
        sum(
            source_lookup[
                image_id
            ].stat().st_size
            for image_id in ids_requiring_copy
        )
    )

    selected_source_gb = (
        selected_source_bytes
        / (1024 ** 3)
    )


    print(
        "\nSelected images requiring copy:",
        len(ids_requiring_copy),
    )

    print(
        "Estimated permanent image size:",
        f"{selected_source_gb:.3f} GB",
    )


    # --------------------------------------------------------
    # 5.7 Copy only sealed images to Google Drive
    #
    # Each copy uses a temporary partial filename so an
    # interrupted copy is never mistaken for a valid image.
    # --------------------------------------------------------

    manifest_by_image_id = (
        manifest
        .set_index("image_id")
    )


    for image_id in tqdm(
        ids_requiring_copy,
        desc="Copying sealed images to Drive",
    ):
        source_path = (
            source_lookup[
                image_id
            ]
        )

        destination_path = Path(
            manifest_by_image_id.loc[
                image_id,
                "local_image_path",
            ]
        )

        destination_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )


        partial_path = (
            destination_path.parent
            / (
                destination_path.name
                + ".partial"
            )
        )


        if partial_path.exists():
            partial_path.unlink()


        try:
            shutil.copyfile(
                source_path,
                partial_path,
            )

            copied_inspection = (
                inspect_image_file(
                    partial_path,
                    compute_hash=False,
                )
            )

            if not copied_inspection[
                "file_valid"
            ]:
                raise RuntimeError(
                    "Copied partial image failed "
                    "integrity inspection."
                )

            os.replace(
                partial_path,
                destination_path,
            )

            downloaded_this_run_ids.add(
                image_id
            )

        except Exception:
            if partial_path.exists():
                partial_path.unlink()

            raise


else:
    print(
        "\nAll sealed target images are already "
        "valid in Google Drive."
    )

    print(
        "Temporary full-dataset download is not required."
    )


# ============================================================
# 6. Final permanent-image integrity audit
# ============================================================

final_audit_rows = []


for row in tqdm(
    manifest.itertuples(
        index=False
    ),
    total=len(manifest),
    desc="Final image integrity audit",
):
    inspection = inspect_image_file(
        row.local_image_path,
        compute_hash=True,
    )

    final_audit_rows.append(
        {
            "acquisition_row": int(
                row.acquisition_row
            ),
            "patient_id": str(
                row.patient_id
            ),
            "split": str(
                row.split
            ),
            "eye_side": str(
                row.eye_side
            ),
            "image_id": str(
                row.image_id
            ),
            "eye_grade": int(
                row.eye_grade
            ),
            "local_image_path": str(
                row.local_image_path
            ),
            "downloaded_this_run": bool(
                row.image_id
                in downloaded_this_run_ids
            ),
            **inspection,
        }
    )


image_integrity_audit = pd.DataFrame(
    final_audit_rows
)


invalid_final_images = (
    image_integrity_audit.loc[
        ~image_integrity_audit[
            "file_valid"
        ]
    ]
    .copy()
)


if len(invalid_final_images) > 0:

    print(
        "\nInvalid final-image examples:"
    )

    display(
        invalid_final_images.head(
            20
        )
    )

    raise RuntimeError(
        f"{len(invalid_final_images)} target images "
        "failed final integrity inspection. "
        "Rerun the cell to reacquire them."
    )


assert (
    image_integrity_audit[
        "file_exists"
    ].all()
)

assert (
    image_integrity_audit[
        "file_valid"
    ].all()
)

assert (
    image_integrity_audit[
        "sha256"
    ].notna().all()
)

assert (
    image_integrity_audit[
        "image_id"
    ].is_unique
)


# ============================================================
# 7. Patient-pair completeness after acquisition
# ============================================================

patient_integrity = (
    image_integrity_audit
    .groupby(
        [
            "patient_id",
            "split",
        ],
        as_index=False,
    )
    .agg(
        images=(
            "image_id",
            "size",
        ),
        valid_images=(
            "file_valid",
            "sum",
        ),
        unique_eyes=(
            "eye_side",
            "nunique",
        ),
    )
)


patient_integrity[
    "complete_valid_pair"
] = (
    patient_integrity[
        "images"
    ].eq(2)
    &
    patient_integrity[
        "valid_images"
    ].eq(2)
    &
    patient_integrity[
        "unique_eyes"
    ].eq(2)
)


assert patient_integrity[
    "complete_valid_pair"
].all()

assert len(
    patient_integrity
) == 2234


# ============================================================
# 8. Acquisition summary
# ============================================================

acquisition_summary = (
    image_integrity_audit
    .groupby(
        "split"
    )
    .agg(
        patients=(
            "patient_id",
            "nunique",
        ),
        images=(
            "image_id",
            "size",
        ),
        valid_images=(
            "file_valid",
            "sum",
        ),
        total_bytes=(
            "file_size_bytes",
            "sum",
        ),
        median_width=(
            "image_width",
            "median",
        ),
        median_height=(
            "image_height",
            "median",
        ),
    )
    .reset_index()
)


acquisition_summary[
    "total_gb"
] = (
    acquisition_summary[
        "total_bytes"
    ]
    / (1024 ** 3)
)


total_permanent_bytes = int(
    image_integrity_audit[
        "file_size_bytes"
    ].sum()
)


total_permanent_gb = (
    total_permanent_bytes
    / (1024 ** 3)
)


# ============================================================
# 9. Save sealed acquisition artifacts
# ============================================================

ACQUIRED_MANIFEST_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Acquired_Image_Manifest_v0.1.csv"
)

IMAGE_INTEGRITY_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Image_Integrity_Audit_v0.1.csv"
)

PATIENT_INTEGRITY_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Acquired_Patient_Integrity_v0.1.csv"
)

ACQUISITION_SUMMARY_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Image_Acquisition_Summary_v0.1.csv"
)

ACQUISITION_DECISION_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Image_Acquisition_Decision_v0.1.json"
)


acquired_manifest = (
    manifest
    .merge(
        image_integrity_audit[
            [
                "image_id",
                "file_size_bytes",
                "image_width",
                "image_height",
                "image_mode",
                "sha256",
                "file_valid",
            ]
        ],
        on="image_id",
        how="left",
        validate="one_to_one",
    )
)


acquired_manifest.to_csv(
    ACQUIRED_MANIFEST_PATH,
    index=False,
)

image_integrity_audit.to_csv(
    IMAGE_INTEGRITY_PATH,
    index=False,
)

patient_integrity.to_csv(
    PATIENT_INTEGRITY_PATH,
    index=False,
)

acquisition_summary.to_csv(
    ACQUISITION_SUMMARY_PATH,
    index=False,
)


acquisition_decision = {
    "decision": (
        "PASS_EYEPACS_IMAGE_ACQUISITION_"
        "ADVANCE_TO_REPRESENTATION_EXTRACTION"
    ),
    "dataset_handle": DATASET_HANDLE,
    "dataset_version": DATASET_VERSION,
    "original_sealed_patients": 2240,
    "availability_filtered_patients": 2234,
    "availability_filtered_images": 4468,
    "development_patients": int(
        patient_integrity.loc[
            patient_integrity[
                "split"
            ].eq("development"),
            "patient_id",
        ].nunique()
    ),
    "validation_patients": int(
        patient_integrity.loc[
            patient_integrity[
                "split"
            ].eq("validation"),
            "patient_id",
        ].nunique()
    ),
    "valid_images": int(
        image_integrity_audit[
            "file_valid"
        ].sum()
    ),
    "invalid_images": int(
        (
            ~image_integrity_audit[
                "file_valid"
            ]
        ).sum()
    ),
    "total_permanent_bytes": (
        total_permanent_bytes
    ),
    "total_permanent_gb": float(
        total_permanent_gb
    ),
    "sha256_recorded_for_all_images": bool(
        image_integrity_audit[
            "sha256"
        ].notna().all()
    ),
    "patients_moved_between_splits": 0,
    "validation_labels_used_for_selection": False,
}


with open(
    ACQUISITION_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        acquisition_decision,
        file,
        indent=2,
    )


# ============================================================
# 10. Delete the temporary full dataset after success
# ============================================================

scratch_deleted = False


if (
    SCRATCH_ROOT.exists()
    and image_integrity_audit[
        "file_valid"
    ].all()
):
    print(
        "\nDeleting temporary full EyePACS dataset "
        "from Colab runtime..."
    )

    shutil.rmtree(
        SCRATCH_ROOT
    )

    scratch_deleted = True


# ============================================================
# 11. Final report
# ============================================================

print(
    "\n================ EYEPACS IMAGE "
    "ACQUISITION SUMMARY ================"
)

display(
    acquisition_summary
)


print(
    "\nPermanent valid images:",
    int(
        image_integrity_audit[
            "file_valid"
        ].sum()
    ),
    "/",
    len(
        image_integrity_audit
    ),
)

print(
    "Permanent selected-image size:",
    f"{total_permanent_gb:.3f} GB",
)

print(
    "Complete patient pairs:",
    int(
        patient_integrity[
            "complete_valid_pair"
        ].sum()
    ),
    "/",
    len(
        patient_integrity
    ),
)

print(
    "Images copied during this run:",
    len(
        downloaded_this_run_ids
    ),
)

print(
    "Temporary full dataset deleted:",
    scratch_deleted,
)


print(
    "\nDecision:"
)

print(
    acquisition_decision[
        "decision"
    ]
)


print("\nSaved:")
print(
    ACQUIRED_MANIFEST_PATH
)
print(
    IMAGE_INTEGRITY_PATH
)
print(
    PATIENT_INTEGRITY_PATH
)
print(
    ACQUISITION_SUMMARY_PATH
)
print(
    ACQUISITION_DECISION_PATH
)


print(
    "\nEyePACS image acquisition completed "
    "and sealed."
)

================ SEALED IMAGE MANIFEST IMPORT ================
Dataset version: 7
Patients: 2234
Images: 4468

Permanent image root:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/02_Dataset_Map/EyePACS_2015_External_Replication/resized_train_cropped


Checking existing Drive images:   0%|          | 0/4468 [00:00<?, ?it/s]


Valid images already present: 0 / 4468
Images requiring acquisition: 4468

Temporary Colab free space: 87.69 GB

Installing or updating KaggleHub...

Nothing from the full archive is being stored permanently in Drive.
Using Colab cache for faster access to the 'diabetic-retinopathy-resized' dataset.

KaggleHub returned:
/kaggle/input/diabetic-retinopathy-resized

Resolved cropped source directory:
/kaggle/input/diabetic-retinopathy-resized/resized_train_cropped
Images detected in source directory: 35108


Indexing cropped source files: 0it [00:00, ?it/s]


Selected images requiring copy: 4468
Estimated permanent image size: 0.775 GB


Copying sealed images to Drive:   0%|          | 0/4468 [00:00<?, ?it/s]

Final image integrity audit:   0%|          | 0/4468 [00:00<?, ?it/s]


Deleting temporary full EyePACS dataset from Colab runtime...

================ EYEPACS IMAGE ACQUISITION SUMMARY ================


,split,patients,images,valid_images,total_bytes,median_width,median_height,total_gb
0,development,1675,3350,3350,623455647,1024.0,920.0,0.580638
1,validation,559,1118,1118,208532283,1024.0,928.0,0.194211



Permanent valid images: 4468 / 4468
Permanent selected-image size: 0.775 GB
Complete patient pairs: 2234 / 2234
Images copied during this run: 4468
Temporary full dataset deleted: True

Decision:
PASS_EYEPACS_IMAGE_ACQUISITION_ADVANCE_TO_REPRESENTATION_EXTRACTION

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/EyePACS_2015_Acquired_Image_Manifest_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/EyePACS_2015_Image_Integrity_Audit_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/EyePACS_2015_Acquired_Patient_Integrity_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/EyePACS_2015_Image_Acquisition_Summary_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observ

In [12]:
#@title 09. Extract and seal clean EyePACS frozen ResNet-50 embeddings

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from tqdm.auto import tqdm

import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision


# ============================================================
# 1. Resolve paths and import sealed acquisition artifacts
# ============================================================

if Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

elif Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")

else:
    raise FileNotFoundError(
        "Google Drive is not mounted. "
        "Please rerun the first notebook cell."
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

EXTERNAL_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "External_Replication_Protocol_v0.1"
)

REPRESENTATION_ROOT = (
    EXTERNAL_ROOT
    / "Frozen_Representation_v0.1"
)

REPRESENTATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


ACQUIRED_MANIFEST_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Acquired_Image_Manifest_v0.1.csv"
)

ACQUISITION_DECISION_PATH = (
    EXTERNAL_ROOT
    / "EyePACS_2015_Image_Acquisition_Decision_v0.1.json"
)


required_paths = [
    ACQUIRED_MANIFEST_PATH,
    ACQUISITION_DECISION_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing sealed acquisition artifacts:\n"
        + "\n".join(
            str(path)
            for path in missing_paths
        )
    )


with open(
    ACQUISITION_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    acquisition_decision = json.load(file)


assert (
    acquisition_decision["decision"]
    ==
    "PASS_EYEPACS_IMAGE_ACQUISITION_"
    "ADVANCE_TO_REPRESENTATION_EXTRACTION"
), acquisition_decision


embedding_index = pd.read_csv(
    ACQUIRED_MANIFEST_PATH
)


required_columns = [
    "acquisition_row",
    "patient_id",
    "split",
    "eye_side",
    "image_id",
    "eye_grade",
    "absolute_grade_gap",
    "left_higher_grade",
    "local_image_path",
    "sha256",
    "file_valid",
]

missing_columns = [
    column
    for column in required_columns
    if column not in embedding_index.columns
]

assert not missing_columns, missing_columns


embedding_index = (
    embedding_index
    .sort_values("acquisition_row")
    .reset_index(drop=True)
    .copy()
)


embedding_index[
    "embedding_row"
] = np.arange(
    len(embedding_index)
)


embedding_index[
    "patient_id"
] = embedding_index[
    "patient_id"
].astype(str)


embedding_index[
    "image_id"
] = embedding_index[
    "image_id"
].astype(str)


assert len(embedding_index) == 4468
assert embedding_index["patient_id"].nunique() == 2234
assert embedding_index["image_id"].is_unique
assert embedding_index["embedding_row"].is_unique
assert embedding_index["file_valid"].astype(bool).all()


missing_image_mask = (
    ~embedding_index[
        "local_image_path"
    ]
    .map(
        lambda path: Path(path).is_file()
    )
)


assert not missing_image_mask.any(), (
    "Some acquired image files are missing."
)


print(
    "================ SEALED EYEPACS "
    "ACQUISITION IMPORT ================"
)

print(
    "Patients:",
    embedding_index["patient_id"].nunique(),
)

print(
    "Images:",
    len(embedding_index),
)

print(
    "\nRows by split:"
)

print(
    embedding_index
    .groupby("split")
    .size()
)


# ============================================================
# 2. Freeze model, weights and official preprocessing
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


weights = (
    ResNet50_Weights.IMAGENET1K_V2
)

preprocess = weights.transforms()


encoder = resnet50(
    weights=weights
)

embedding_dimension = (
    encoder.fc.in_features
)

encoder.fc = nn.Identity()

encoder.eval()


for parameter in encoder.parameters():
    parameter.requires_grad = False


encoder = encoder.to(device)


trainable_parameters = sum(
    parameter.numel()
    for parameter in encoder.parameters()
    if parameter.requires_grad
)


assert embedding_dimension == 2048
assert trainable_parameters == 0


print(
    "\n================ FROZEN ENCODER "
    "================"
)

print(
    "PyTorch:",
    torch.__version__,
)

print(
    "TorchVision:",
    torchvision.__version__,
)

print(
    "Device:",
    device,
)

if device.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

print(
    "Encoder: ResNet-50 / ImageNet-1K V2"
)

print(
    "Embedding dimension:",
    embedding_dimension,
)

print(
    "Trainable parameters:",
    trainable_parameters,
)


# ============================================================
# 3. Dataset
# ============================================================

class EyePACSEmbeddingDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform,
    ):
        self.dataframe = (
            dataframe.reset_index(
                drop=True
            )
        )

        self.transform = transform


    def __len__(self):
        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        index,
    ):
        row = self.dataframe.iloc[
            index
        ]

        image_path = Path(
            row[
                "local_image_path"
            ]
        )

        embedding_row = int(
            row[
                "embedding_row"
            ]
        )

        try:
            with Image.open(
                image_path
            ) as image:
                image = image.convert(
                    "RGB"
                )

                image_tensor = (
                    self.transform(
                        image
                    )
                )

        except Exception as error:
            raise RuntimeError(
                "Failed to load image:\n"
                f"row={embedding_row}\n"
                f"path={image_path}"
            ) from error

        return (
            image_tensor,
            embedding_row,
        )


dataset = EyePACSEmbeddingDataset(
    dataframe=embedding_index,
    transform=preprocess,
)


# ============================================================
# 4. DataLoader
# ============================================================

if device.type == "cuda":
    BATCH_SIZE = 64
    NUM_WORKERS = 2

else:
    BATCH_SIZE = 16
    NUM_WORKERS = 2


loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(
        device.type == "cuda"
    ),
    persistent_workers=(
        NUM_WORKERS > 0
    ),
)


print(
    "\nBatches:",
    len(loader),
)

print(
    "Batch size:",
    BATCH_SIZE,
)

print(
    "Workers:",
    NUM_WORKERS,
)


# ============================================================
# 5. Output paths
# ============================================================

EMBEDDING_PATH = (
    REPRESENTATION_ROOT
    / (
        "EyePACS_2015_ResNet50_ImageNet1K_V2_"
        "Clean_Embeddings_float32.npy"
    )
)

EMBEDDING_INDEX_PATH = (
    REPRESENTATION_ROOT
    / (
        "EyePACS_2015_ResNet50_ImageNet1K_V2_"
        "Clean_Embedding_Index.csv"
    )
)

EMBEDDING_AUDIT_PATH = (
    REPRESENTATION_ROOT
    / (
        "EyePACS_2015_ResNet50_ImageNet1K_V2_"
        "Clean_Embedding_Audit.csv"
    )
)

EMBEDDING_DECISION_PATH = (
    REPRESENTATION_ROOT
    / (
        "EyePACS_2015_Clean_Embedding_"
        "Decision_v0.1.json"
    )
)

ENVIRONMENT_PATH = (
    REPRESENTATION_ROOT
    / (
        "EyePACS_2015_Clean_Embedding_"
        "Environment_v0.1.json"
    )
)


# ============================================================
# 6. Extract or load cached embeddings
# ============================================================

cache_complete = (
    EMBEDDING_PATH.is_file()
    and EMBEDDING_INDEX_PATH.is_file()
)


if cache_complete:

    print(
        "\nExisting clean embedding cache detected."
    )

    print(
        "Loading:"
    )

    print(
        EMBEDDING_PATH
    )

    embeddings = np.load(
        EMBEDDING_PATH,
        allow_pickle=False,
    )

    cached_index = pd.read_csv(
        EMBEDDING_INDEX_PATH
    )

    assert len(
        cached_index
    ) == len(
        embedding_index
    )

    assert (
        cached_index[
            "image_id"
        ].astype(str).tolist()
        ==
        embedding_index[
            "image_id"
        ].astype(str).tolist()
    )


else:

    print(
        "\nExtracting clean frozen embeddings..."
    )

    embeddings = np.empty(
        (
            len(embedding_index),
            embedding_dimension,
        ),
        dtype=np.float32,
    )


    with torch.inference_mode():

        for (
            image_batch,
            embedding_rows,
        ) in tqdm(
            loader,
            desc=(
                "EyePACS clean ResNet-50"
            ),
        ):

            image_batch = image_batch.to(
                device,
                non_blocking=(
                    device.type == "cuda"
                ),
            )


            if device.type == "cuda":

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    batch_embeddings = (
                        encoder(
                            image_batch
                        )
                    )

            else:
                batch_embeddings = encoder(
                    image_batch
                )


            batch_embeddings = (
                batch_embeddings
                .float()
                .cpu()
                .numpy()
            )


            embedding_rows = (
                embedding_rows
                .numpy()
            )


            embeddings[
                embedding_rows
            ] = batch_embeddings


    np.save(
        EMBEDDING_PATH,
        embeddings,
        allow_pickle=False,
    )

    embedding_index.to_csv(
        EMBEDDING_INDEX_PATH,
        index=False,
    )


    print(
        "\nClean embedding extraction completed."
    )

    print(
        "Saved embeddings:"
    )

    print(
        EMBEDDING_PATH
    )

    print(
        "Saved index:"
    )

    print(
        EMBEDDING_INDEX_PATH
    )


# ============================================================
# 7. Embedding integrity audit
# ============================================================

assert embeddings.shape == (
    len(embedding_index),
    embedding_dimension,
), embeddings.shape


assert embeddings.dtype == np.float32


finite_values = bool(
    np.isfinite(
        embeddings
    ).all()
)


assert finite_values


embedding_norms = np.linalg.norm(
    embeddings,
    axis=1,
)


dimension_standard_deviation = (
    embeddings.std(
        axis=0
    )
)


near_zero_dimensions = int(
    (
        dimension_standard_deviation
        < 1e-8
    ).sum()
)


duplicate_embedding_rows = int(
    pd.DataFrame(
        embeddings
    ).duplicated().sum()
)


embedding_audit = pd.DataFrame(
    [
        {
            "condition": "clean",
            "rows": int(
                embeddings.shape[0]
            ),
            "dimensions": int(
                embeddings.shape[1]
            ),
            "dtype": str(
                embeddings.dtype
            ),
            "finite_values": (
                finite_values
            ),
            "norm_min": float(
                embedding_norms.min()
            ),
            "norm_mean": float(
                embedding_norms.mean()
            ),
            "norm_max": float(
                embedding_norms.max()
            ),
            "dimension_sd_min": float(
                dimension_standard_deviation.min()
            ),
            "dimension_sd_median": float(
                np.median(
                    dimension_standard_deviation
                )
            ),
            "dimension_sd_max": float(
                dimension_standard_deviation.max()
            ),
            "near_zero_variance_dimensions": (
                near_zero_dimensions
            ),
            "duplicate_embedding_rows": (
                duplicate_embedding_rows
            ),
        }
    ]
)


embedding_audit.to_csv(
    EMBEDDING_AUDIT_PATH,
    index=False,
)


assert embedding_norms.min() > 0
assert (
    dimension_standard_deviation
    > 1e-8
).any()


# ============================================================
# 8. Record environment and sealed decision
# ============================================================

manifest_fingerprint_columns = [
    "embedding_row",
    "patient_id",
    "split",
    "eye_side",
    "image_id",
    "eye_grade",
    "sha256",
]


manifest_fingerprint_text = (
    embedding_index[
        manifest_fingerprint_columns
    ]
    .to_csv(
        index=False
    )
)


manifest_sha256 = hashlib.sha256(
    manifest_fingerprint_text.encode(
        "utf-8"
    )
).hexdigest()


environment_record = {
    "python": sys.version,
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "torchvision": torchvision.__version__,
    "device": str(device),
    "gpu": (
        torch.cuda.get_device_name(0)
        if device.type == "cuda"
        else None
    ),
    "encoder": (
        "torchvision.models.resnet50"
    ),
    "weights": (
        "ResNet50_Weights.IMAGENET1K_V2"
    ),
    "preprocessing": (
        "ResNet50_Weights."
        "IMAGENET1K_V2.transforms()"
    ),
    "embedding_dimension": (
        embedding_dimension
    ),
    "trainable_parameters": (
        trainable_parameters
    ),
    "manifest_sha256": (
        manifest_sha256
    ),
}


with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_record,
        file,
        indent=2,
    )


embedding_decision = {
    "decision": (
        "PASS_EYEPACS_CLEAN_FROZEN_"
        "EMBEDDINGS_ADVANCE_TO_"
        "SEALED_CONDITION_EXTRACTION"
    ),
    "condition": "clean",
    "patients": int(
        embedding_index[
            "patient_id"
        ].nunique()
    ),
    "images": int(
        len(
            embedding_index
        )
    ),
    "development_patients": int(
        embedding_index.loc[
            embedding_index[
                "split"
            ].eq("development"),
            "patient_id",
        ].nunique()
    ),
    "validation_patients": int(
        embedding_index.loc[
            embedding_index[
                "split"
            ].eq("validation"),
            "patient_id",
        ].nunique()
    ),
    "embedding_shape": [
        int(
            embeddings.shape[0]
        ),
        int(
            embeddings.shape[1]
        ),
    ],
    "embedding_dtype": str(
        embeddings.dtype
    ),
    "finite_values": (
        finite_values
    ),
    "near_zero_variance_dimensions": (
        near_zero_dimensions
    ),
    "duplicate_embedding_rows": (
        duplicate_embedding_rows
    ),
    "manifest_sha256": (
        manifest_sha256
    ),
    "validation_labels_used_for_selection": (
        False
    ),
}


with open(
    EMBEDDING_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        embedding_decision,
        file,
        indent=2,
    )


# ============================================================
# 9. Final report
# ============================================================

print(
    "\n================ CLEAN EMBEDDING "
    "AUDIT ================"
)

display(
    embedding_audit
)


print(
    "\nEmbedding shape:",
    embeddings.shape,
)

print(
    "Embedding dtype:",
    embeddings.dtype,
)

print(
    "Finite values:",
    finite_values,
)

print(
    "Near-zero-variance dimensions:",
    near_zero_dimensions,
)

print(
    "Duplicate embedding rows:",
    duplicate_embedding_rows,
)


print(
    "\nDecision:"
)

print(
    embedding_decision[
        "decision"
    ]
)


print(
    "\nSaved:"
)

print(
    EMBEDDING_PATH
)

print(
    EMBEDDING_INDEX_PATH
)

print(
    EMBEDDING_AUDIT_PATH
)

print(
    EMBEDDING_DECISION_PATH
)

print(
    ENVIRONMENT_PATH
)


print(
    "\nEyePACS clean frozen representation "
    "extraction completed and sealed."
)

================ SEALED EYEPACS ACQUISITION IMPORT ================
Patients: 2234
Images: 4468

Rows by split:
split
development    3350
validation     1118
dtype: int64
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 137MB/s]



================ FROZEN ENCODER ================
PyTorch: 2.11.0+cpu
TorchVision: 0.26.0+cpu
Device: cpu
Encoder: ResNet-50 / ImageNet-1K V2
Embedding dimension: 2048
Trainable parameters: 0

Batches: 280
Batch size: 16
Workers: 2

Extracting clean frozen embeddings...


EyePACS clean ResNet-50:   0%|          | 0/280 [00:00<?, ?it/s]


Clean embedding extraction completed.
Saved embeddings:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/Frozen_Representation_v0.1/EyePACS_2015_ResNet50_ImageNet1K_V2_Clean_Embeddings_float32.npy
Saved index:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/Frozen_Representation_v0.1/EyePACS_2015_ResNet50_ImageNet1K_V2_Clean_Embedding_Index.csv

================ CLEAN EMBEDDING AUDIT ================


,condition,rows,dimensions,dtype,finite_values,norm_min,norm_mean,norm_max,dimension_sd_min,dimension_sd_median,dimension_sd_max,near_zero_variance_dimensions,duplicate_embedding_rows
0,clean,4468,2048,float32,True,8.257221,12.400569,17.190952,0.000298,0.036195,1.42159,0,0



Embedding shape: (4468, 2048)
Embedding dtype: float32
Finite values: True
Near-zero-variance dimensions: 0
Duplicate embedding rows: 0

Decision:
PASS_EYEPACS_CLEAN_FROZEN_EMBEDDINGS_ADVANCE_TO_SEALED_CONDITION_EXTRACTION

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/Frozen_Representation_v0.1/EyePACS_2015_ResNet50_ImageNet1K_V2_Clean_Embeddings_float32.npy
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/Frozen_Representation_v0.1/EyePACS_2015_ResNet50_ImageNet1K_V2_Clean_Embedding_Index.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/Frozen_Representation_v0.1/EyePACS_2015_ResNet50_ImageNet1K_V2_Clean_Embedding_Audit.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/Frozen_Rep

In [13]:
#@title 10. Sealed clean EyePACS paired-difference external replication

from pathlib import Path

import hashlib
import json
import platform
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn

from scipy.stats import beta, binomtest

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. Resolve paths
# ============================================================

if Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

elif Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")

else:
    raise FileNotFoundError(
        "Google Drive is not mounted. "
        "Please rerun the notebook bootstrap cell."
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

EXTERNAL_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "External_Replication_Protocol_v0.1"
)

REPRESENTATION_ROOT = (
    EXTERNAL_ROOT
    / "Frozen_Representation_v0.1"
)

CLEAN_REPLICATION_ROOT = (
    EXTERNAL_ROOT
    / "Clean_Locality_Replication_v0.1"
)

CLEAN_REPLICATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


EMBEDDING_PATH = (
    REPRESENTATION_ROOT
    / (
        "EyePACS_2015_ResNet50_ImageNet1K_V2_"
        "Clean_Embeddings_float32.npy"
    )
)

EMBEDDING_INDEX_PATH = (
    REPRESENTATION_ROOT
    / (
        "EyePACS_2015_ResNet50_ImageNet1K_V2_"
        "Clean_Embedding_Index.csv"
    )
)

EMBEDDING_DECISION_PATH = (
    REPRESENTATION_ROOT
    / (
        "EyePACS_2015_Clean_Embedding_"
        "Decision_v0.1.json"
    )
)


required_paths = [
    EMBEDDING_PATH,
    EMBEDDING_INDEX_PATH,
    EMBEDDING_DECISION_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing sealed clean-representation artifacts:\n"
        + "\n".join(
            str(path)
            for path in missing_paths
        )
    )


# ============================================================
# 2. Import and verify sealed clean embeddings
# ============================================================

with open(
    EMBEDDING_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    embedding_decision = json.load(file)


assert (
    embedding_decision["decision"]
    ==
    "PASS_EYEPACS_CLEAN_FROZEN_"
    "EMBEDDINGS_ADVANCE_TO_"
    "SEALED_CONDITION_EXTRACTION"
), embedding_decision


embeddings = np.load(
    EMBEDDING_PATH,
    allow_pickle=False,
)

embedding_index = pd.read_csv(
    EMBEDDING_INDEX_PATH
)


required_index_columns = [
    "embedding_row",
    "patient_id",
    "split",
    "eye_side",
    "image_id",
    "eye_grade",
    "fellow_eye_grade",
    "absolute_grade_gap",
    "left_higher_grade",
]

missing_index_columns = [
    column
    for column in required_index_columns
    if column not in embedding_index.columns
]

assert not missing_index_columns, (
    missing_index_columns
)


embedding_index = (
    embedding_index
    .sort_values("embedding_row")
    .reset_index(drop=True)
    .copy()
)


embedding_index["patient_id"] = (
    embedding_index[
        "patient_id"
    ].astype(str)
)

embedding_index["image_id"] = (
    embedding_index[
        "image_id"
    ].astype(str)
)

embedding_index["eye_side"] = (
    embedding_index[
        "eye_side"
    ].astype(str)
    .str.lower()
)


assert embeddings.shape == (
    4468,
    2048,
)

assert embeddings.dtype == np.float32
assert np.isfinite(embeddings).all()

assert len(embedding_index) == embeddings.shape[0]

assert np.array_equal(
    embedding_index[
        "embedding_row"
    ].astype(int).to_numpy(),
    np.arange(
        len(embedding_index)
    ),
)

assert embedding_index[
    "image_id"
].is_unique

assert (
    embedding_index
    .groupby("patient_id")
    .size()
    .eq(2)
    .all()
)

assert (
    embedding_index
    .groupby("patient_id")[
        "eye_side"
    ]
    .nunique()
    .eq(2)
    .all()
)


print(
    "================ SEALED CLEAN "
    "REPRESENTATION IMPORT ================"
)

print(
    "Embedding shape:",
    embeddings.shape,
)

print(
    "Patients:",
    embedding_index[
        "patient_id"
    ].nunique(),
)

print(
    "\nImages by split:"
)

print(
    embedding_index
    .groupby("split")
    .size()
)


# ============================================================
# 3. Construct one signed left-minus-right vector per patient
# ============================================================

left_index = (
    embedding_index.loc[
        embedding_index[
            "eye_side"
        ].eq("left"),
        [
            "patient_id",
            "split",
            "embedding_row",
            "image_id",
            "eye_grade",
            "fellow_eye_grade",
            "absolute_grade_gap",
            "left_higher_grade",
        ],
    ]
    .rename(
        columns={
            "embedding_row": (
                "left_embedding_row"
            ),
            "image_id": (
                "left_image_id"
            ),
            "eye_grade": (
                "left_grade"
            ),
            "fellow_eye_grade": (
                "right_grade_from_left_row"
            ),
            "absolute_grade_gap": (
                "left_absolute_grade_gap"
            ),
            "left_higher_grade": (
                "left_higher_grade_from_left_row"
            ),
        }
    )
    .copy()
)


right_index = (
    embedding_index.loc[
        embedding_index[
            "eye_side"
        ].eq("right"),
        [
            "patient_id",
            "split",
            "embedding_row",
            "image_id",
            "eye_grade",
            "fellow_eye_grade",
            "absolute_grade_gap",
            "left_higher_grade",
        ],
    ]
    .rename(
        columns={
            "embedding_row": (
                "right_embedding_row"
            ),
            "image_id": (
                "right_image_id"
            ),
            "eye_grade": (
                "right_grade"
            ),
            "fellow_eye_grade": (
                "left_grade_from_right_row"
            ),
            "absolute_grade_gap": (
                "right_absolute_grade_gap"
            ),
            "left_higher_grade": (
                "left_higher_grade_from_right_row"
            ),
        }
    )
    .copy()
)


assert left_index[
    "patient_id"
].is_unique

assert right_index[
    "patient_id"
].is_unique


patient_pairs = (
    left_index
    .merge(
        right_index,
        on=[
            "patient_id",
            "split",
        ],
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        [
            "split",
            "patient_id",
        ]
    )
    .reset_index(drop=True)
)


assert len(patient_pairs) == 2234

assert (
    patient_pairs[
        "patient_id"
    ].nunique()
    == 2234
)


assert np.array_equal(
    patient_pairs[
        "left_grade"
    ].astype(int).to_numpy(),
    patient_pairs[
        "left_grade_from_right_row"
    ].astype(int).to_numpy(),
)

assert np.array_equal(
    patient_pairs[
        "right_grade"
    ].astype(int).to_numpy(),
    patient_pairs[
        "right_grade_from_left_row"
    ].astype(int).to_numpy(),
)

assert np.array_equal(
    patient_pairs[
        "left_absolute_grade_gap"
    ].astype(int).to_numpy(),
    patient_pairs[
        "right_absolute_grade_gap"
    ].astype(int).to_numpy(),
)

assert np.array_equal(
    patient_pairs[
        "left_higher_grade_from_left_row"
    ].astype(int).to_numpy(),
    patient_pairs[
        "left_higher_grade_from_right_row"
    ].astype(int).to_numpy(),
)


patient_pairs[
    "absolute_grade_gap"
] = (
    patient_pairs[
        "left_grade"
    ].astype(int)
    -
    patient_pairs[
        "right_grade"
    ].astype(int)
).abs()


patient_pairs[
    "left_higher_grade"
] = (
    patient_pairs[
        "left_grade"
    ].astype(int)
    >
    patient_pairs[
        "right_grade"
    ].astype(int)
).astype(int)


assert (
    patient_pairs[
        "absolute_grade_gap"
    ] > 0
).all()


assert np.array_equal(
    patient_pairs[
        "absolute_grade_gap"
    ].astype(int).to_numpy(),
    patient_pairs[
        "left_absolute_grade_gap"
    ].astype(int).to_numpy(),
)

assert np.array_equal(
    patient_pairs[
        "left_higher_grade"
    ].astype(int).to_numpy(),
    patient_pairs[
        "left_higher_grade_from_left_row"
    ].astype(int).to_numpy(),
)


left_rows = (
    patient_pairs[
        "left_embedding_row"
    ].astype(int).to_numpy()
)

right_rows = (
    patient_pairs[
        "right_embedding_row"
    ].astype(int).to_numpy()
)


pair_difference_embeddings = (
    embeddings[left_rows]
    -
    embeddings[right_rows]
).astype(
    np.float32,
    copy=False,
)


pair_labels = (
    patient_pairs[
        "left_higher_grade"
    ].astype(int).to_numpy()
)


assert pair_difference_embeddings.shape == (
    2234,
    2048,
)

assert np.isfinite(
    pair_difference_embeddings
).all()

assert set(
    np.unique(pair_labels)
) == {
    0,
    1,
}


# ============================================================
# 4. Freeze development and validation cohorts
# ============================================================

development_mask = (
    patient_pairs[
        "split"
    ].eq("development")
    .to_numpy()
)

validation_mask = (
    patient_pairs[
        "split"
    ].eq("validation")
    .to_numpy()
)


X_development = (
    pair_difference_embeddings[
        development_mask
    ]
)

y_development = (
    pair_labels[
        development_mask
    ]
)

development_groups = (
    patient_pairs.loc[
        development_mask,
        "patient_id",
    ]
    .astype(str)
    .to_numpy()
)


X_validation = (
    pair_difference_embeddings[
        validation_mask
    ]
)

y_validation = (
    pair_labels[
        validation_mask
    ]
)

validation_pairs = (
    patient_pairs.loc[
        validation_mask
    ]
    .copy()
    .reset_index(drop=True)
)


assert X_development.shape == (
    1675,
    2048,
)

assert X_validation.shape == (
    559,
    2048,
)

assert len(
    set(
        development_groups
    )
    &
    set(
        validation_pairs[
            "patient_id"
        ].astype(str)
    )
) == 0


print(
    "\n================ PAIRED-DIFFERENCE "
    "DATA AUDIT ================"
)

print(
    "Development patients:",
    len(
        X_development
    ),
)

print(
    "Validation patients:",
    len(
        X_validation
    ),
)

print(
    "Patient overlap:",
    0,
)

print(
    "\nLeft-higher prevalence:"
)

print(
    patient_pairs
    .groupby("split")[
        "left_higher_grade"
    ]
    .agg(
        [
            "count",
            "mean",
        ]
    )
)


# ============================================================
# 5. Symmetric paired-difference construction
#
# For each patient:
#   x = left embedding - right embedding
#   y = 1 when left grade is higher
#
# Add the reversed orientation:
#  -x with label 1-y
#
# The same patient remains grouped during CV.
# ============================================================

def make_symmetric_pairs(
    features,
    labels,
    groups,
):
    features = np.asarray(
        features,
        dtype=np.float32,
    )

    labels = np.asarray(
        labels,
        dtype=int,
    )

    groups = np.asarray(
        groups
    )

    symmetric_features = np.concatenate(
        [
            features,
            -features,
        ],
        axis=0,
    )

    symmetric_labels = np.concatenate(
        [
            labels,
            1 - labels,
        ],
        axis=0,
    )

    symmetric_groups = np.concatenate(
        [
            groups,
            groups,
        ],
        axis=0,
    )

    return (
        symmetric_features,
        symmetric_labels,
        symmetric_groups,
    )


(
    X_development_symmetric,
    y_development_symmetric,
    development_groups_symmetric,
) = make_symmetric_pairs(
    X_development,
    y_development,
    development_groups,
)


assert X_development_symmetric.shape == (
    3350,
    2048,
)

assert (
    y_development_symmetric.mean()
    == 0.5
)


# ============================================================
# 6. Pre-register model and hyperparameter grid
# ============================================================

RANDOM_SEED = 20260720
NUMBER_OF_FOLDS = 5

C_GRID = [
    1e-5,
    3e-5,
    1e-4,
    3e-4,
    1e-3,
    3e-3,
    1e-2,
    3e-2,
    1e-1,
]


def make_probe(
    regularization_c,
):
    return Pipeline(
        steps=[
            (
                "standard_scaler",
                StandardScaler(),
            ),
            (
                "logistic_regression",
                LogisticRegression(
                    C=float(
                        regularization_c
                    ),
                    penalty="l2",
                    solver="liblinear",
                    max_iter=5000,
                    random_state=(
                        RANDOM_SEED
                    ),
                ),
            ),
        ]
    )


cross_validator = StratifiedGroupKFold(
    n_splits=NUMBER_OF_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED,
)


# ============================================================
# 7. Development-only grouped cross-validation
# ============================================================

cv_rows = []


for regularization_c in C_GRID:

    fold_accuracies = []
    fold_balanced_accuracies = []
    fold_aucs = []

    for (
        fold_number,
        (
            train_indices,
            held_out_indices,
        ),
    ) in enumerate(
        cross_validator.split(
            X_development_symmetric,
            y_development_symmetric,
            groups=(
                development_groups_symmetric
            ),
        ),
        start=1,
    ):

        train_groups = set(
            development_groups_symmetric[
                train_indices
            ]
        )

        held_out_groups = set(
            development_groups_symmetric[
                held_out_indices
            ]
        )

        assert len(
            train_groups
            &
            held_out_groups
        ) == 0


        fold_model = make_probe(
            regularization_c
        )

        fold_model.fit(
            X_development_symmetric[
                train_indices
            ],
            y_development_symmetric[
                train_indices
            ],
        )


        fold_scores = (
            fold_model.decision_function(
                X_development_symmetric[
                    held_out_indices
                ]
            )
        )

        fold_predictions = (
            fold_scores > 0
        ).astype(int)


        fold_accuracy = accuracy_score(
            y_development_symmetric[
                held_out_indices
            ],
            fold_predictions,
        )

        fold_balanced_accuracy = (
            balanced_accuracy_score(
                y_development_symmetric[
                    held_out_indices
                ],
                fold_predictions,
            )
        )

        fold_auc = roc_auc_score(
            y_development_symmetric[
                held_out_indices
            ],
            fold_scores,
        )


        fold_accuracies.append(
            fold_accuracy
        )

        fold_balanced_accuracies.append(
            fold_balanced_accuracy
        )

        fold_aucs.append(
            fold_auc
        )


    cv_rows.append(
        {
            "C": float(
                regularization_c
            ),
            "mean_grouped_cv_accuracy": float(
                np.mean(
                    fold_accuracies
                )
            ),
            "sd_grouped_cv_accuracy": float(
                np.std(
                    fold_accuracies,
                    ddof=1,
                )
            ),
            "mean_grouped_cv_balanced_accuracy": float(
                np.mean(
                    fold_balanced_accuracies
                )
            ),
            "sd_grouped_cv_balanced_accuracy": float(
                np.std(
                    fold_balanced_accuracies,
                    ddof=1,
                )
            ),
            "mean_grouped_cv_auc": float(
                np.mean(
                    fold_aucs
                )
            ),
            "sd_grouped_cv_auc": float(
                np.std(
                    fold_aucs,
                    ddof=1,
                )
            ),
        }
    )


cv_results = pd.DataFrame(
    cv_rows
)


cv_results = (
    cv_results
    .sort_values(
        by=[
            "mean_grouped_cv_auc",
            "mean_grouped_cv_balanced_accuracy",
            "C",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


selected_c = float(
    cv_results.iloc[0][
        "C"
    ]
)


print(
    "\n================ DEVELOPMENT-ONLY "
    "GROUPED CV ================"
)

display(
    cv_results
)

print(
    "Selected C:",
    selected_c,
)


# ============================================================
# 8. Development out-of-fold performance with selected C
# ============================================================

development_oof_scores_symmetric = np.full(
    len(
        X_development_symmetric
    ),
    np.nan,
    dtype=float,
)


for (
    train_indices,
    held_out_indices,
) in cross_validator.split(
    X_development_symmetric,
    y_development_symmetric,
    groups=(
        development_groups_symmetric
    ),
):

    fold_model = make_probe(
        selected_c
    )

    fold_model.fit(
        X_development_symmetric[
            train_indices
        ],
        y_development_symmetric[
            train_indices
        ],
    )

    development_oof_scores_symmetric[
        held_out_indices
    ] = (
        fold_model.decision_function(
            X_development_symmetric[
                held_out_indices
            ]
        )
    )


assert np.isfinite(
    development_oof_scores_symmetric
).all()


# Use only the original left-minus-right orientation.
development_oof_scores = (
    development_oof_scores_symmetric[
        :len(
            X_development
        )
    ]
)


development_oof_predictions = (
    development_oof_scores > 0
).astype(int)


development_oof_accuracy = float(
    accuracy_score(
        y_development,
        development_oof_predictions,
    )
)

development_oof_balanced_accuracy = float(
    balanced_accuracy_score(
        y_development,
        development_oof_predictions,
    )
)

development_oof_auc = float(
    roc_auc_score(
        y_development,
        development_oof_scores,
    )
)


# ============================================================
# 9. Fit final development model
# ============================================================

final_probe = make_probe(
    selected_c
)

final_probe.fit(
    X_development_symmetric,
    y_development_symmetric,
)


# ============================================================
# 10. One-time sealed validation evaluation
# ============================================================

validation_scores = (
    final_probe.decision_function(
        X_validation
    )
)

validation_predictions = (
    validation_scores > 0
).astype(int)


validation_correct = (
    validation_predictions
    == y_validation
)


number_correct = int(
    validation_correct.sum()
)

number_validation_patients = int(
    len(
        y_validation
    )
)


validation_accuracy = float(
    accuracy_score(
        y_validation,
        validation_predictions,
    )
)

validation_balanced_accuracy = float(
    balanced_accuracy_score(
        y_validation,
        validation_predictions,
    )
)

validation_directional_auc = float(
    roc_auc_score(
        y_validation,
        validation_scores,
    )
)


# ============================================================
# 11. Exact binomial inference
# ============================================================

def clopper_pearson_interval(
    successes,
    trials,
    confidence_level=0.95,
):
    alpha = (
        1.0
        - confidence_level
    )

    if successes == 0:
        lower = 0.0

    else:
        lower = float(
            beta.ppf(
                alpha / 2,
                successes,
                trials
                - successes
                + 1,
            )
        )


    if successes == trials:
        upper = 1.0

    else:
        upper = float(
            beta.ppf(
                1
                - alpha / 2,
                successes + 1,
                trials - successes,
            )
        )


    return (
        lower,
        upper,
    )


(
    validation_accuracy_ci_lower,
    validation_accuracy_ci_upper,
) = clopper_pearson_interval(
    number_correct,
    number_validation_patients,
)


validation_one_sided_p = float(
    binomtest(
        k=number_correct,
        n=number_validation_patients,
        p=0.5,
        alternative="greater",
    ).pvalue
)


# ============================================================
# 12. Pre-registered external-replication gate
#
# Primary:
# - Exact one-sided binomial p < 0.05
#
# Supporting:
# - Accuracy above chance
# - Balanced accuracy above chance
# - Directional AUC above chance
# ============================================================

primary_significance_gate = bool(
    validation_one_sided_p
    < 0.05
)

accuracy_direction_gate = bool(
    validation_accuracy
    > 0.5
)

balanced_accuracy_gate = bool(
    validation_balanced_accuracy
    > 0.5
)

directional_auc_gate = bool(
    validation_directional_auc
    > 0.5
)


passes_clean_replication = bool(
    primary_significance_gate
    and accuracy_direction_gate
    and balanced_accuracy_gate
    and directional_auc_gate
)


replication_decision = (
    "PASS_EYEPACS_CLEAN_LOCALITY_REPLICATION_"
    "ADVANCE_TO_SEALED_PERTURBATION_EXTRACTION"
    if passes_clean_replication
    else
    "FAIL_EYEPACS_CLEAN_LOCALITY_REPLICATION_GATE"
)


# ============================================================
# 13. Validation prediction table
# ============================================================

validation_predictions_table = (
    validation_pairs[
        [
            "patient_id",
            "split",
            "left_image_id",
            "right_image_id",
            "left_grade",
            "right_grade",
            "absolute_grade_gap",
            "left_higher_grade",
        ]
    ]
    .copy()
)


validation_predictions_table[
    "true_left_higher_grade"
] = y_validation


validation_predictions_table[
    "decision_score_left_minus_right"
] = validation_scores


validation_predictions_table[
    "predicted_left_higher_grade"
] = validation_predictions


validation_predictions_table[
    "correct"
] = validation_correct


validation_predictions_table[
    "signed_grade_difference"
] = (
    validation_predictions_table[
        "left_grade"
    ].astype(int)
    -
    validation_predictions_table[
        "right_grade"
    ].astype(int)
)


# ============================================================
# 14. Grade-gap subgroup audit
# ============================================================

grade_gap_rows = []


validation_predictions_table[
    "grade_gap_group"
] = np.where(
    validation_predictions_table[
        "absolute_grade_gap"
    ].astype(int)
    == 1,
    "gap_1",
    "gap_2_or_more",
)


for (
    subgroup_name,
    subgroup_df,
) in validation_predictions_table.groupby(
    "grade_gap_group"
):

    subgroup_true = (
        subgroup_df[
            "true_left_higher_grade"
        ].astype(int).to_numpy()
    )

    subgroup_predicted = (
        subgroup_df[
            "predicted_left_higher_grade"
        ].astype(int).to_numpy()
    )

    subgroup_scores = (
        subgroup_df[
            "decision_score_left_minus_right"
        ].to_numpy()
    )


    subgroup_correct = int(
        (
            subgroup_true
            == subgroup_predicted
        ).sum()
    )

    subgroup_n = int(
        len(
            subgroup_df
        )
    )

    (
        subgroup_ci_lower,
        subgroup_ci_upper,
    ) = clopper_pearson_interval(
        subgroup_correct,
        subgroup_n,
    )


    subgroup_auc = (
        float(
            roc_auc_score(
                subgroup_true,
                subgroup_scores,
            )
        )
        if len(
            np.unique(
                subgroup_true
            )
        ) == 2
        else np.nan
    )


    grade_gap_rows.append(
        {
            "subgroup": (
                subgroup_name
            ),
            "patients": (
                subgroup_n
            ),
            "correct_pairs": (
                subgroup_correct
            ),
            "accuracy": float(
                subgroup_correct
                / subgroup_n
            ),
            "accuracy_ci_lower": (
                subgroup_ci_lower
            ),
            "accuracy_ci_upper": (
                subgroup_ci_upper
            ),
            "directional_auc": (
                subgroup_auc
            ),
        }
    )


grade_gap_results = pd.DataFrame(
    grade_gap_rows
)


# ============================================================
# 15. Comparison with sealed DeepDRiD discovery result
# ============================================================

DEEPDRID_REFERENCE_CORRECT = 31
DEEPDRID_REFERENCE_TOTAL = 45

DEEPDRID_REFERENCE_ACCURACY = (
    DEEPDRID_REFERENCE_CORRECT
    / DEEPDRID_REFERENCE_TOTAL
)

DEEPDRID_REFERENCE_DIRECTIONAL_AUC = (
    0.7530364372469636
)

DEEPDRID_REFERENCE_ONE_SIDED_P = (
    0.008047180015637421
)


evidence_table = pd.DataFrame(
    [
        {
            "cohort": (
                "DeepDRiD discovery"
            ),
            "patients": (
                DEEPDRID_REFERENCE_TOTAL
            ),
            "correct_pairs": (
                DEEPDRID_REFERENCE_CORRECT
            ),
            "pairwise_accuracy": (
                DEEPDRID_REFERENCE_ACCURACY
            ),
            "accuracy_ci_lower": (
                0.5335089700878829
            ),
            "accuracy_ci_upper": (
                0.818341196066763
            ),
            "balanced_accuracy": (
                0.688259109311741
            ),
            "directional_auc": (
                DEEPDRID_REFERENCE_DIRECTIONAL_AUC
            ),
            "exact_one_sided_p": (
                DEEPDRID_REFERENCE_ONE_SIDED_P
            ),
            "role": (
                "sealed discovery reference"
            ),
        },
        {
            "cohort": (
                "EyePACS external validation"
            ),
            "patients": (
                number_validation_patients
            ),
            "correct_pairs": (
                number_correct
            ),
            "pairwise_accuracy": (
                validation_accuracy
            ),
            "accuracy_ci_lower": (
                validation_accuracy_ci_lower
            ),
            "accuracy_ci_upper": (
                validation_accuracy_ci_upper
            ),
            "balanced_accuracy": (
                validation_balanced_accuracy
            ),
            "directional_auc": (
                validation_directional_auc
            ),
            "exact_one_sided_p": (
                validation_one_sided_p
            ),
            "role": (
                "independent external replication"
            ),
        },
    ]
)


# ============================================================
# 16. Save artifacts
# ============================================================

CV_RESULTS_PATH = (
    CLEAN_REPLICATION_ROOT
    / (
        "EyePACS_Clean_PairedDifference_"
        "Grouped_CV_v0.1.csv"
    )
)

VALIDATION_PREDICTIONS_PATH = (
    CLEAN_REPLICATION_ROOT
    / (
        "EyePACS_Clean_Validation_"
        "PairedDifference_Predictions_v0.1.csv"
    )
)

GRADE_GAP_RESULTS_PATH = (
    CLEAN_REPLICATION_ROOT
    / (
        "EyePACS_Clean_Validation_"
        "GradeGap_Subgroups_v0.1.csv"
    )
)

EVIDENCE_TABLE_PATH = (
    CLEAN_REPLICATION_ROOT
    / (
        "DeepDRiD_EyePACS_Clean_"
        "Replication_Evidence_v0.1.csv"
    )
)

MODEL_PATH = (
    CLEAN_REPLICATION_ROOT
    / (
        "EyePACS_Clean_Symmetric_"
        "PairedDifference_Probe_v0.1.joblib"
    )
)

SUMMARY_PATH = (
    CLEAN_REPLICATION_ROOT
    / (
        "EyePACS_Clean_Locality_"
        "Replication_Summary_v0.1.json"
    )
)

DECISION_PATH = (
    CLEAN_REPLICATION_ROOT
    / (
        "EyePACS_Clean_Locality_"
        "Replication_Decision_v0.1.json"
    )
)

ENVIRONMENT_PATH = (
    CLEAN_REPLICATION_ROOT
    / (
        "EyePACS_Clean_Locality_"
        "Replication_Environment_v0.1.json"
    )
)


cv_results.to_csv(
    CV_RESULTS_PATH,
    index=False,
)

validation_predictions_table.to_csv(
    VALIDATION_PREDICTIONS_PATH,
    index=False,
)

grade_gap_results.to_csv(
    GRADE_GAP_RESULTS_PATH,
    index=False,
)

evidence_table.to_csv(
    EVIDENCE_TABLE_PATH,
    index=False,
)

joblib.dump(
    final_probe,
    MODEL_PATH,
)


summary = {
    "analysis": (
        "EyePACS clean symmetric "
        "paired-difference locality replication"
    ),
    "encoder": (
        "ResNet-50 / ImageNet-1K V2"
    ),
    "representation": (
        "frozen clean 2048-dimensional embeddings"
    ),
    "development_patients": int(
        len(
            X_development
        )
    ),
    "validation_patients": (
        number_validation_patients
    ),
    "selected_C": (
        selected_c
    ),
    "development_oof_accuracy": (
        development_oof_accuracy
    ),
    "development_oof_balanced_accuracy": (
        development_oof_balanced_accuracy
    ),
    "development_oof_auc": (
        development_oof_auc
    ),
    "validation_correct_pairs": (
        number_correct
    ),
    "validation_pairwise_accuracy": (
        validation_accuracy
    ),
    "validation_accuracy_exact_two_sided_95ci": [
        validation_accuracy_ci_lower,
        validation_accuracy_ci_upper,
    ],
    "validation_balanced_accuracy": (
        validation_balanced_accuracy
    ),
    "validation_directional_auc": (
        validation_directional_auc
    ),
    "validation_exact_one_sided_binomial_p": (
        validation_one_sided_p
    ),
    "validation_labels_used_for_model_selection": (
        False
    ),
}


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        indent=2,
    )


decision_record = {
    "decision": (
        replication_decision
    ),
    "passes_clean_replication": (
        passes_clean_replication
    ),
    "primary_significance_gate": (
        primary_significance_gate
    ),
    "accuracy_direction_gate": (
        accuracy_direction_gate
    ),
    "balanced_accuracy_gate": (
        balanced_accuracy_gate
    ),
    "directional_auc_gate": (
        directional_auc_gate
    ),
    "validation_patients": (
        number_validation_patients
    ),
    "validation_correct_pairs": (
        number_correct
    ),
    "validation_pairwise_accuracy": (
        validation_accuracy
    ),
    "validation_accuracy_ci_lower": (
        validation_accuracy_ci_lower
    ),
    "validation_accuracy_ci_upper": (
        validation_accuracy_ci_upper
    ),
    "validation_balanced_accuracy": (
        validation_balanced_accuracy
    ),
    "validation_directional_auc": (
        validation_directional_auc
    ),
    "validation_exact_one_sided_p": (
        validation_one_sided_p
    ),
    "deepdrid_reference_accuracy": (
        DEEPDRID_REFERENCE_ACCURACY
    ),
    "deepdrid_reference_directional_auc": (
        DEEPDRID_REFERENCE_DIRECTIONAL_AUC
    ),
    "validation_labels_used_for_selection": (
        False
    ),
}


with open(
    DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_record,
        file,
        indent=2,
    )


environment_record = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "model": (
        "StandardScaler + L2 LogisticRegression"
    ),
    "solver": "liblinear",
    "number_of_cv_folds": (
        NUMBER_OF_FOLDS
    ),
    "random_seed": (
        RANDOM_SEED
    ),
    "C_grid": C_GRID,
    "feature_orientation": (
        "left_embedding_minus_right_embedding"
    ),
    "symmetric_training": True,
    "patient_grouped_cross_validation": True,
}


with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_record,
        file,
        indent=2,
    )


# ============================================================
# 17. Final report
# ============================================================

print(
    "\n================ EYEPACS CLEAN "
    "EXTERNAL REPLICATION ================"
)

print(
    "Selected development-only C:",
    selected_c,
)

print(
    "\nDevelopment grouped OOF:"
)

print(
    "Accuracy:",
    f"{development_oof_accuracy:.4f}",
)

print(
    "Balanced accuracy:",
    f"{development_oof_balanced_accuracy:.4f}",
)

print(
    "Directional AUC:",
    f"{development_oof_auc:.4f}",
)


print(
    "\nSealed EyePACS validation:"
)

print(
    "Correct pairs:",
    f"{number_correct}/{number_validation_patients}",
)

print(
    "Pairwise accuracy:",
    f"{validation_accuracy:.4f}",
)

print(
    "Exact 95% CI:",
    (
        f"[{validation_accuracy_ci_lower:.4f}, "
        f"{validation_accuracy_ci_upper:.4f}]"
    ),
)

print(
    "Balanced accuracy:",
    f"{validation_balanced_accuracy:.4f}",
)

print(
    "Directional AUC:",
    f"{validation_directional_auc:.4f}",
)

print(
    "Exact one-sided binomial p:",
    f"{validation_one_sided_p:.8g}",
)


print(
    "\nGrade-gap subgroups:"
)

display(
    grade_gap_results
)


print(
    "\nDiscovery-versus-replication evidence:"
)

display(
    evidence_table
)


print(
    "\nDecision:"
)

print(
    replication_decision
)


print(
    "\nSaved:"
)

print(
    CV_RESULTS_PATH
)

print(
    VALIDATION_PREDICTIONS_PATH
)

print(
    GRADE_GAP_RESULTS_PATH
)

print(
    EVIDENCE_TABLE_PATH
)

print(
    MODEL_PATH
)

print(
    SUMMARY_PATH
)

print(
    DECISION_PATH
)

print(
    ENVIRONMENT_PATH
)


print(
    "\nEyePACS clean external replication "
    "completed and sealed."
)

================ SEALED CLEAN REPRESENTATION IMPORT ================
Embedding shape: (4468, 2048)
Patients: 2234

Images by split:
split
development    3350
validation     1118
dtype: int64

================ PAIRED-DIFFERENCE DATA AUDIT ================
Development patients: 1675
Validation patients: 559
Patient overlap: 0

Left-higher prevalence:
             count      mean
split                       
development   1675  0.524179
validation     559  0.525939

================ DEVELOPMENT-ONLY GROUPED CV ================


,C,mean_grouped_cv_accuracy,sd_grouped_cv_accuracy,mean_grouped_cv_balanced_accuracy,sd_grouped_cv_balanced_accuracy,mean_grouped_cv_auc,sd_grouped_cv_auc
0,0.01000,0.539701,0.028741,0.539701,0.028741,0.561657,0.028170
1,0.00300,0.539701,0.031549,0.539701,0.031549,0.561440,0.024253
2,0.03000,0.546866,0.024288,0.546866,0.024288,0.561037,0.029413
3,0.00010,0.542090,0.015453,0.542090,0.015453,0.560560,0.019010
4,0.00030,0.543881,0.023731,0.543881,0.023731,0.560558,0.014087
5,0.00100,0.537313,0.033574,0.537313,0.033574,0.560470,0.018026
6,0.10000,0.540896,0.022178,0.540896,0.022178,0.559476,0.029758
7,0.00003,0.540896,0.029806,0.540896,0.029806,0.556304,0.029740
8,0.00001,0.528358,0.032081,0.528358,0.032081,0.550521,0.039212


Selected C: 0.01

================ EYEPACS CLEAN EXTERNAL REPLICATION ================
Selected development-only C: 0.01

Development grouped OOF:
Accuracy: 0.5397
Balanced accuracy: 0.5374
Directional AUC: 0.5595

Sealed EyePACS validation:
Correct pairs: 289/559
Pairwise accuracy: 0.5170
Exact 95% CI: [0.4747, 0.5591]
Balanced accuracy: 0.5151
Directional AUC: 0.5197
Exact one-sided binomial p: 0.22324765

Grade-gap subgroups:


,subgroup,patients,correct_pairs,accuracy,accuracy_ci_lower,accuracy_ci_upper,directional_auc
0,gap_1,370,183,0.494595,0.442520,0.546757,0.497625
1,gap_2_or_more,189,106,0.560847,0.486966,0.632788,0.570083



Discovery-versus-replication evidence:


,cohort,patients,correct_pairs,pairwise_accuracy,accuracy_ci_lower,accuracy_ci_upper,balanced_accuracy,directional_auc,exact_one_sided_p,role
0,DeepDRiD discovery,45,31,0.688889,0.533509,0.818341,0.688259,0.753036,0.008047,sealed discovery reference
1,EyePACS external validation,559,289,0.516995,0.474679,0.559130,0.515133,0.519728,0.223248,independent external replication



Decision:
FAIL_EYEPACS_CLEAN_LOCALITY_REPLICATION_GATE

Saved:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/Clean_Locality_Replication_v0.1/EyePACS_Clean_PairedDifference_Grouped_CV_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/Clean_Locality_Replication_v0.1/EyePACS_Clean_Validation_PairedDifference_Predictions_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/Clean_Locality_Replication_v0.1/EyePACS_Clean_Validation_GradeGap_Subgroups_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Replication_Protocol_v0.1/Clean_Locality_Replication_v0.1/DeepDRiD_EyePACS_Clean_Replication_Evidence_v0.1.csv
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/External_Repl